# RAG v2 (Markdown Table Parser)

## 프로젝트 개요
B2G 입찰 컨설팅 스타트업 **입찰메이트**의 사내 RAG 시스템.
100개의 RFP(제안요청서) 문서에서 핵심 정보를 추출하고, 질의응답을 제공한다.

## 파이프라인 구조
```
[원본 HWP/PDF] → [Parser] → [Retriever] → [LangGraph] → [답변]
     │               │             │              │
     │          텍스트 추출    벡터 검색     Self-Corrective RAG
     │          필드 추출     Chroma DB      답변 생성 + 출처 인용
     │          청킹          메타 필터      관련성 평가 + 재검색
```

## 모듈 구성
| 모듈 | 역할 |
| :--- | :--- |
| **Parser** | HWP/HWPX/PDF → 텍스트 추출, 정제, 청킹 |
| **Retriever** | 임베딩, Chroma 적재, 유사도 검색 |
| **Prompt** | 답변 생성 / 관련성 평가 / 질문 재작성 프롬프트 |
| **LangGraph** | Self-Corrective RAG 그래프 오케스트레이션 |

## 1. 환경 설정

In [1]:
# 필요 라이브러리 설치 (최초 1회)
# !pip install python-hwpx pyhwp lxml pymupdf4llm
# !pip install langchain langchain-openai langchain-chroma langchain-text-splitters
# !pip install langgraph chromadb pandas openpyxl python-dotenv
# !pip install rank-bm25 kiwipiepy nest-asyncio
# !pip install flashrank  # Cross-Encoder Re-ranker (경량 ONNX, PyTorch 불필요)
# !pip install streamlit plotly  # 웹 대시보드

### 설치 가이드 (Install Guide)

| 패키지 그룹 | 설치 명령어 | 용도 |
|:---|:---|:---|
| **문서 파싱** | `pip install python-hwpx pyhwp lxml pymupdf4llm` | HWP/HWPX/PDF 파싱 |
| **LangChain** | `pip install langchain langchain-openai langchain-chroma langchain-text-splitters` | RAG 프레임워크 |
| **오케스트레이션** | `pip install langgraph chromadb pandas openpyxl python-dotenv` | 그래프·벡터DB·데이터 |
| **하이브리드 검색** | `pip install rank-bm25 kiwipiepy nest-asyncio` | BM25 + 한국어 형태소 |
| **Re-ranker** | `pip install flashrank` | Cross-Encoder 재정렬 (ONNX) |
| **웹 대시보드** | `pip install streamlit plotly` | Streamlit UI + 차트 |

**환경 변수** (프로젝트 루트에 `.env` 파일 생성):
```
OPENAI_API_KEY=sk-...
```

**Streamlit 실행**:
```bash
cd src && streamlit run app.py
```

In [2]:
import os
import re
import json
import time
import pickle
import traceback
import zipfile
from pathlib import Path
from typing import List, Optional
from datetime import datetime

import nest_asyncio
import pandas as pd
import pymupdf4llm
from dotenv import load_dotenv
from lxml import etree
from pydantic import BaseModel, Field

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langgraph.graph import START, END, StateGraph
from typing import TypedDict
import hashlib

from hwp5.xmlmodel import Hwp5File
from hwp5.binmodel.tagid51_para_text import ParaText

# BM25 하이브리드 검색 + 한국어 형태소 분석
from rank_bm25 import BM25Okapi
from kiwipiepy import Kiwi

# Cross-Encoder Re-ranker (경량 ONNX 기반)
try:
    from flashrank import Ranker, RerankRequest
    FLASHRANK_AVAILABLE = True
except ImportError:
    FLASHRANK_AVAILABLE = False
    print("flashrank 미설치 — Re-ranker 비활성화 (pip install flashrank)")

# Jupyter 환경 비동기 호환 (이미 실행 중인 이벤트 루프에서 await 사용 가능)
nest_asyncio.apply()

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
flashrank 미설치 — Re-ranker 비활성화 (pip install flashrank)


In [3]:
# .env 로드 및 경로 설정
PROJECT_ROOT = Path(os.getcwd()).parent
env_path = PROJECT_ROOT / ".env"
load_dotenv(dotenv_path=env_path)

DATA_DIR = PROJECT_ROOT / "원본 데이터"
FILES_DIR = DATA_DIR / "files"
CSV_PATH = DATA_DIR / "data_list.csv"
PERSIST_DIR = str(PROJECT_ROOT / "chroma_db_v2")
BM25_INDEX_PATH = str(PROJECT_ROOT / "bm25_index.pkl")
ERROR_LOG_PATH = str(PROJECT_ROOT / "error_log.json")
HASH_CACHE_PATH = str(PROJECT_ROOT / "file_hashes.json")

assert os.environ.get("OPENAI_API_KEY"), (
    "OPENAI_API_KEY가 없습니다. .env 파일을 확인해주세요."
)

print(f"프로젝트 루트: {PROJECT_ROOT}")
print(f".env 로드: {env_path.exists()}")
print(f"CSV 존재: {CSV_PATH.exists()}")
print(f"BM25 인덱스 경로: {BM25_INDEX_PATH}")
print(f"에러 로그 경로: {ERROR_LOG_PATH}")
print(f"해시 캐시 경로: {HASH_CACHE_PATH}")
print(f"OPENAI_API_KEY: 설정됨")

프로젝트 루트: c:\Users\User\Desktop\중급 프로젝트
.env 로드: True
CSV 존재: True
BM25 인덱스 경로: c:\Users\User\Desktop\중급 프로젝트\bm25_index.pkl
에러 로그 경로: c:\Users\User\Desktop\중급 프로젝트\error_log.json
해시 캐시 경로: c:\Users\User\Desktop\중급 프로젝트\file_hashes.json
OPENAI_API_KEY: 설정됨


---
## 2. Parser — 문서 파싱

원본 RFP 문서(HWP/HWPX/PDF)를 읽어 구조화된 텍스트로 변환한다.
```
[원본 파일] → [텍스트 추출] → [정제] → [필드 추출] → [청킹] → [Document 리스트]
```

In [4]:
# CSV 메타데이터 로드
df_raw = pd.read_csv(CSV_PATH, encoding="utf-8")

META_COLUMNS = ["공고 번호", "사업명", "사업 금액", "발주 기관",
               "공개 일자", "입찰 참여 마감일", "사업 요약", "파일형식", "파일명"]

print(f"전체 문서 수: {len(df_raw)}")
print("파일형식 분포:")
print(df_raw["파일형식"].value_counts())
print()
print(f"사용 메타데이터 컬럼: {META_COLUMNS}")
print("[참고] 텍스트는 CSV가 아닌 실제 파일(HWP/PDF)에서 파싱합니다.")


전체 문서 수: 100
파일형식 분포:
파일형식
hwp    96
pdf     4
Name: count, dtype: int64

사용 메타데이터 컬럼: ['공고 번호', '사업명', '사업 금액', '발주 기관', '공개 일자', '입찰 참여 마감일', '사업 요약', '파일형식', '파일명']
[참고] 텍스트는 CSV가 아닌 실제 파일(HWP/PDF)에서 파싱합니다.


In [5]:
# --- HWP 5.x 파서 (pyhwp 기반 OLE2 바이너리, 마크다운 테이블 변환) ---

WIDE_TABLE_THRESHOLD = 6  # 이 열 수를 초과하면 압축 텍스트로 변환


def _table_to_markdown(cells: dict, max_row: int, max_col: int) -> str:
    """표 셀 데이터를 마크다운 테이블 또는 압축 텍스트로 변환한다.

    - 빈 테이블 → 스킵
    - 1행/1열 레이아웃 테이블 → 일반 텍스트
    - 넓은 테이블(>6열) → 빈 셀 제거 후 압축 텍스트
    - 일반 테이블(≤6열) → 마크다운 테이블
    """
    if max_row < 0 or max_col < 0:
        return ""

    # 전체 텍스트가 비어있으면 스킵
    all_text = "".join("".join(v) for v in cells.values()).strip()
    if not all_text:
        return ""

    num_rows = max_row + 1
    num_cols = max_col + 1

    # 1행 또는 1열 → 레이아웃용이므로 일반 텍스트
    if num_cols == 1 or num_rows == 1:
        parts = []
        for r in range(num_rows):
            for c in range(num_cols):
                text = " ".join(cells.get((r, c), [])).strip()
                if text:
                    parts.append(text)
        return "\n".join(parts)

    # ── 넓은 테이블(>6열) → 압축 텍스트 ──
    if num_cols > WIDE_TABLE_THRESHOLD:
        return _wide_table_to_text(cells, num_rows, num_cols)

    # ── 일반 테이블(≤6열) → 마크다운 ──
    return _narrow_table_to_markdown(cells, num_rows, num_cols)


def _wide_table_to_text(cells: dict, num_rows: int, num_cols: int) -> str:
    """넓은 테이블을 빈 셀 제거 후 압축 텍스트로 변환한다."""
    lines = []
    for r in range(num_rows):
        row_values = []
        for c in range(num_cols):
            text = " ".join(cells.get((r, c), [])).strip()
            if text:
                row_values.append(text)
        if row_values:
            lines.append(" | ".join(row_values))
    return "\n".join(lines)


def _narrow_table_to_markdown(cells: dict, num_rows: int, num_cols: int) -> str:
    """일반 테이블(≤6열)을 마크다운 테이블로 변환한다."""
    rows = []
    for r in range(num_rows):
        row = []
        for c in range(num_cols):
            cell_text = " ".join(cells.get((r, c), [])).strip()
            cell_text = cell_text.replace("|", "/")
            cell_text = cell_text.replace("\n", " ")
            row.append(cell_text if cell_text else " ")
        rows.append(row)

    # 빈 첫 행 → 내용 있는 행을 헤더로 승격
    if all(cell.strip() == "" for cell in rows[0]) and len(rows) > 1:
        for idx in range(len(rows)):
            if any(cell.strip() for cell in rows[idx]):
                rows = rows[idx:]
                break
        else:
            return ""

    lines = []
    lines.append("| " + " | ".join(rows[0]) + " |")
    lines.append("| " + " | ".join(["---"] * num_cols) + " |")
    for row in rows[1:]:
        lines.append("| " + " | ".join(row) + " |")
    return "\n".join(lines)


def extract_text_from_hwp(file_path: str) -> str:
    """HWP 5.x (OLE2) 파일에서 pyhwp로 텍스트를 추출한다.

    - 섹션별/이벤트별 try-except로 일부 손상이 있어도 나머지를 추출
    - 일반 표(≤6열): 마크다운 테이블로 구조 보존
    - 넓은 표(>6열): 빈 셀 제거 후 압축 텍스트
    """
    try:
        hwpfile = Hwp5File(file_path)
    except Exception as e:
        return f"[HWP 파싱 오류: 파일 열기 실패 — {type(e).__name__}: {e}]"

    output_parts = []
    section_errors = []

    try:
        section_indexes = list(hwpfile.bodytext.section_indexes())
    except Exception as e:
        return f"[HWP 파싱 오류: 섹션 목록 조회 실패 — {type(e).__name__}: {e}]"

    for sec_idx in section_indexes:
        try:
            sec = hwpfile.bodytext.section(sec_idx)
        except Exception as e:
            section_errors.append(f"섹션 {sec_idx} 로드 실패: {e}")
            continue

        in_table = False
        table_cells = {}
        max_row = -1
        max_col = -1
        cur_row = 0
        cur_col = 0

        try:
            events = list(sec.modelevents())
        except Exception as e:
            section_errors.append(f"섹션 {sec_idx} 이벤트 조회 실패: {e}")
            continue

        for event_type, payload in events:
            try:
                if not isinstance(payload, tuple) or len(payload) < 1:
                    continue

                evt_name = event_type.__name__
                model_class = payload[0]
                model_name = (
                    model_class.__name__
                    if hasattr(model_class, "__name__")
                    else ""
                )
                attrs = payload[1] if len(payload) > 1 else {}

                # 테이블 시작
                if model_name == "TableControl" and evt_name == "STARTEVENT":
                    in_table = True
                    table_cells = {}
                    max_row = -1
                    max_col = -1
                    continue

                # 테이블 종료 → 변환 후 빈 줄로 감싸기
                if model_name == "TableControl" and evt_name == "ENDEVENT":
                    if table_cells:
                        try:
                            converted = _table_to_markdown(
                                table_cells, max_row, max_col
                            )
                            if converted:
                                output_parts.append("")
                                output_parts.append(converted)
                                output_parts.append("")
                        except Exception:
                            # 테이블 변환 실패 시 원본 텍스트로 폴백
                            fallback = " ".join(
                                " ".join(v) for v in table_cells.values()
                            ).strip()
                            if fallback:
                                output_parts.append(fallback)
                    in_table = False
                    table_cells = {}
                    continue

                # 셀 위치 추적
                if (
                    model_name == "TableCell"
                    and evt_name == "STARTEVENT"
                    and in_table
                ):
                    cur_row = attrs.get("row", 0)
                    cur_col = attrs.get("col", 0)
                    colspan = attrs.get("colspan", 1)
                    rowspan = attrs.get("rowspan", 1)
                    max_row = max(max_row, cur_row + rowspan - 1)
                    max_col = max(max_col, cur_col + colspan - 1)
                    continue

                # 텍스트 추출 (STARTEVENT만)
                if evt_name == "STARTEVENT" and model_class == ParaText:
                    text_chunks = []
                    for item in attrs.get("chunks", []):
                        if isinstance(item, tuple) and len(item) == 2:
                            _, data = item
                            if isinstance(data, str) and data.strip():
                                text_chunks.append(data.strip())

                    if not text_chunks:
                        continue

                    joined = " ".join(text_chunks)

                    if in_table:
                        table_cells.setdefault(
                            (cur_row, cur_col), []
                        ).append(joined)
                    else:
                        output_parts.append(joined)

            except Exception:
                continue  # 개별 이벤트 오류 → 무시하고 다음 이벤트 처리

    result = "\n".join(output_parts)
    if section_errors and not result.strip():
        return f"[HWP 파싱 오류: {'; '.join(section_errors)}]"
    return result


# --- HWPX 파서 (ZIP 내부 XML) ---

def extract_text_from_hwpx(file_path: str) -> str:
    """HWPX 파일에서 본문 텍스트를 추출한다."""
    texts = []
    try:
        with zipfile.ZipFile(file_path, "r") as zf:
            section_files = sorted(
                n for n in zf.namelist()
                if "section" in n.lower() and n.endswith(".xml")
            )
            for sf in section_files:
                try:
                    root = etree.fromstring(zf.read(sf))
                    for node in root.iter():
                        if node.text and node.text.strip():
                            texts.append(node.text.strip())
                except etree.XMLSyntaxError as e:
                    texts.append(f"[HWPX 섹션 파싱 오류: {sf} — {e}]")
                    continue
    except zipfile.BadZipFile:
        return "[HWPX 파싱 오류: 손상된 ZIP 파일]"
    except Exception as e:
        return f"[HWPX 파싱 오류: {type(e).__name__}: {e}]"
    return "\n".join(texts)


# --- PDF 파서 ---

MAX_PDF_SIZE_MB = 100


def extract_text_from_pdf(file_path: str) -> str:
    """PDF 파일에서 pymupdf4llm으로 텍스트를 추출한다."""
    try:
        file_size = Path(file_path).stat().st_size
        if file_size > MAX_PDF_SIZE_MB * 1024 * 1024:
            size_mb = file_size / (1024 * 1024)
            return f"[PDF 파싱 오류: 파일 크기 초과 ({size_mb:.0f}MB > {MAX_PDF_SIZE_MB}MB)]"
        return pymupdf4llm.to_markdown(file_path)
    except Exception as e:
        return f"[PDF 파싱 오류: {type(e).__name__}: {e}]"


print("파서 함수 정의 완료: HWP (섹션별 안전 파싱), HWPX (섹션별 폴백), PDF (크기 제한)")

파서 함수 정의 완료: HWP (섹션별 안전 파싱), HWPX (섹션별 폴백), PDF (크기 제한)


In [6]:
# --- 텍스트 정제 ---

PAGE_NUMBER_PATTERNS = [
    r"^\s*-\s*\d+\s*-\s*$",
    r"^\s*--\s*\d+\s*--\s*$",
    r"^\s*\(\d+\)\s*$",
    r"^\s*\d+\s*/\s*\d+\s*$",
    r"^\s*페이지\s*\d+\s*$",
    r"^\s*-\d{1,3}\s*$",           # PDF 마크다운 페이지번호: -1, -296
]
HEADER_FOOTER_PATTERNS = [
    r"^\s*제안요청서\s*$",
    r"^\s*- \d+ -\s*$",
]

# 목차 라인 패턴: "Ⅰ. 제목 - 1" 또는 "1. 제목 - 1" 형태
TOC_LINE_PATTERN = re.compile(
    r"^\s*[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩⅰⅱⅲⅳⅴⅵⅶⅷⅸⅹ\d]+\.?\s*.+\s*-\s*\d+\s*$"
)

# 목차 내 부록 항목 패턴: "[별지 1] 제목  35"
TOC_APPENDIX_LINE_PATTERN = re.compile(
    r"^\s*[\[【<]?\s*(?:별지|붙임|서식|별표|첨부)\s*[\d제호]*\s*[\]】>]?\s*.+\s+\d{1,3}\s*$"
)

# PDF 목차 점선 패턴: "1. 사업개요 ···················· 4"
TOC_DOTTED_PATTERN = re.compile(
    r"^.+[·∙⋅‧·]{5,}.+$"
)

# 부록 섹션 시작 헤더 패턴 (실제 서식/양식 내용의 시작)
APPENDIX_HEADER_PATTERNS = [
    # ── HWP 공통 ──
    re.compile(r"^\s*[\[【]\s*별지[\s\d제호]*[\]】]"),
    re.compile(r"^\s*[\[【]\s*붙임[\s\d]*[\]】]"),
    re.compile(r"^\s*[\[【]\s*서식[\s\d제호]*[\]】]"),
    re.compile(r"^\s*[\[【]\s*별표[\s\d]*[\]】]"),
    re.compile(r"^\s*[\[【]\s*첨부[\s\d]*[\]】]"),
    re.compile(r"^\s*<서식[\s\d제호]*>"),
    re.compile(r"^\s*별지\s*제?\s*\d+\s*호?\s*서식"),
    re.compile(r"^\s*[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+\.?\s*(?:첨\s*부|부\s*록|별\s*첨)"),
    re.compile(r"^\s*입찰\s*및\s*제안서\s*관련\s*서식"),
    # ── PDF 마크다운 (볼드 마커 제거 후 매칭) ──
    re.compile(r"^\s*\[별지서식[\s\d]*\]"),
    re.compile(r"^\s*\[붙임\s*\d+\]"),
]


def _strip_md_bold(text: str) -> str:
    """마크다운 볼드 마커(**) 와 여분 공백을 제거한다. [** 붙임 **3] → [붙임 3]"""
    text = text.replace("**", "")
    text = re.sub(r"\[\s+", "[", text)
    text = re.sub(r"\s+\]", "]", text)
    return text


def clean_pdf_artifacts(text: str) -> str:
    """PDF→마크다운 변환 시 발생하는 아티팩트를 정리한다."""
    lines = text.split("\n")
    cleaned = []
    for line in lines:
        stripped = line.strip()

        # 1) 목차 점선 줄 제거: "1. 사업개요 ···················· 4"
        if TOC_DOTTED_PATTERN.match(stripped):
            continue

        # 2) 빈 마크다운 테이블 헤더 제거: |Col1|Col2|Col3|
        if re.match(r"^\|(?:Col\d+\|)+$", stripped):
            continue

        # 2-1) 셀 내 Col1, Col2 등 플레이스홀더를 공백으로 치환
        if "Col" in line:
            line = re.sub(r"Col\d+", " ", line)
            stripped = line.strip()

        # 3) 테이블 구분선만 있는 줄 제거: |---|---|---|
        if re.match(r"^\|(?:\s*-+\s*\|)+$", stripped):
            continue

        # 4) 볼드 마커 정규화: **[** 붙임 **3]** → [붙임 3]
        if "**" in stripped:
            line = _strip_md_bold(line)

        cleaned.append(line)

    return "\n".join(cleaned)


def remove_toc_section(text: str) -> str:
    """목차(Table of Contents) 섹션을 제거한다."""
    lines = text.split("\n")
    result_lines = []
    in_toc = False
    toc_line_count = 0

    for i, line in enumerate(lines):
        stripped = line.strip()

        # 목차 시작 감지: "목 차", "목차", "# **목  차**" (PDF 마크다운)
        clean_stripped = _strip_md_bold(stripped).replace("#", "").strip()
        if re.match(r"^목\s*차\s*$", clean_stripped):
            in_toc = True
            toc_line_count = 0
            continue

        if in_toc:
            if TOC_LINE_PATTERN.match(stripped) or TOC_APPENDIX_LINE_PATTERN.match(stripped):
                toc_line_count += 1
                continue
            if not stripped:
                continue
            if re.match(r"^\s*서식\s*\d+[\.\s].+\d{1,3}\s*$", stripped):
                toc_line_count += 1
                continue
            if toc_line_count >= 3:
                in_toc = False
                result_lines.append(line)
            elif toc_line_count == 0:
                in_toc = False
                result_lines.append(line)
            continue

        result_lines.append(line)

    return "\n".join(result_lines)


def remove_appendix_section(text: str) -> str:
    """문서 후반부의 부록/서식/별지/붙임 섹션을 제거한다."""
    lines = text.split("\n")
    total_chars = len(text)
    if total_chars == 0:
        return text

    min_position = int(total_chars * 0.4)
    running_pos = 0

    for i, line in enumerate(lines):
        if running_pos < min_position:
            running_pos += len(line) + 1
            continue

        stripped = line.strip()

        # 부록 헤더 매칭
        is_appendix_header = any(p.match(stripped) for p in APPENDIX_HEADER_PATTERNS)
        if not is_appendix_header:
            running_pos += len(line) + 1
            continue

        # 목차 항목인지 확인 (다음 줄이 페이지 번호만)
        next_stripped = lines[i + 1].strip() if i + 1 < len(lines) else ""
        if re.match(r"^\d{1,3}$", next_stripped):
            running_pos += len(line) + 1
            continue

        return "\n".join(lines[:i])

    return text


def _is_pdf_markdown(text: str) -> bool:
    """텍스트가 PDF→마크다운 변환 결과인지 간단히 판별한다."""
    sample = text[:3000]
    md_indicators = sum([
        sample.count("|") > 10,
        sample.count("**") > 3,
        "```" in sample or "# " in sample,
        bool(re.search(r"-\d{1,3}\s*$", sample, re.MULTILINE)),
    ])
    return md_indicators >= 2


def clean_text(raw_text: str) -> str:
    """페이지 번호/머리말 제거, 목차 제거, 부록 제거, 공백 정규화."""
    if not raw_text or not raw_text.strip():
        return ""

    text = raw_text

    # Step 0: PDF 마크다운 아티팩트 정리 (점선 목차, 빈 테이블, 볼드 정규화)
    if _is_pdf_markdown(text):
        text = clean_pdf_artifacts(text)

    # Step 1: 목차 섹션 제거
    text = remove_toc_section(text)

    # Step 2: 부록/서식/별지 섹션 제거
    text = remove_appendix_section(text)

    # Step 3: 페이지 번호/헤더 패턴 + 숫자만 있는 줄 제거
    lines = text.split("\n")
    all_patterns = PAGE_NUMBER_PATTERNS + HEADER_FOOTER_PATTERNS
    compiled = [re.compile(p, re.MULTILINE) for p in all_patterns]
    cleaned_lines = []
    for line in lines:
        if any(p.match(line) for p in compiled):
            continue
        if re.match(r"^\s*\d{1,3}\s*$", line.strip()):
            continue
        cleaned = re.sub(r"[ \t]+", " ", line).strip()
        if cleaned:
            cleaned_lines.append(cleaned)
    result = "\n".join(cleaned_lines)
    return re.sub(r"\n{3,}", "\n\n", result)


print("clean_text 정의 완료 (HWP + PDF 마크다운 정제)")


clean_text 정의 완료 (HWP + PDF 마크다운 정제)


In [7]:
# --- 핵심 정보 필드 추출 ---

BID_TYPE_KEYWORDS = [
    "협상에 의한 계약", "제한경쟁입찰", "일반경쟁입찰",
    "수의계약", "지명경쟁입찰", "협상에의한계약",
    "제한경쟁", "일반경쟁",
]


def extract_bid_type(text: str) -> str:
    for kw in BID_TYPE_KEYWORDS:
        if kw in text:
            return kw
    return ""


def extract_qualification(text: str) -> str:
    for pat in [
        r"참가\s*자격[^\n]*\n((?:.*\n){1,10})",
        r"입찰\s*참가\s*자격[^\n]*\n((?:.*\n){1,10})",
        r"참여\s*자격[^\n]*\n((?:.*\n){1,10})",
    ]:
        m = re.search(pat, text)
        if m:
            return m.group(1).strip()
    return ""


def extract_fields(row: pd.Series, cleaned_text: str) -> dict:
    """CSV 메타 + 텍스트 분석으로 핵심 필드를 추출한다."""
    budget_raw = row.get("사업 금액")
    if pd.notna(budget_raw):
        bv = int(budget_raw)
        budget_str = (
            f"{bv / 1e8:.1f}억원" if bv >= 1e8
            else f"{bv / 1e4:.0f}만원" if bv >= 1e4
            else f"{bv:,}원"
        )
    else:
        budget_str = ""
    deadline = str(row.get("입찰 참여 마감일", ""))
    if deadline == "nan":
        deadline = ""
    return {
        "사업명": str(row.get("사업명", "")),
        "발주기관": str(row.get("발주 기관", "")),
        "사업예산": budget_str,
        "입찰방식": extract_bid_type(cleaned_text),
        "제출기한": deadline,
        "참가자격": extract_qualification(cleaned_text),
    }


print("필드 추출 함수 정의 완료")

필드 추출 함수 정의 완료


In [8]:
# --- 청킹 (Table-aware: 마크다운 테이블 구조 보존) ---

CHUNK_SIZE = 800
CHUNK_OVERLAP = 200
SEPARATORS = ["\n\n", "\n", ". ", "다. ", "요. ", "함. ", " "]

# 마크다운 테이블 패턴
_TABLE_ROW_RE = re.compile(r"^\|.+\|$")
_TABLE_SEP_RE = re.compile(r"^\|\s*[-:]+[\s|:-]*\|$")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=SEPARATORS,
    length_function=len,
)


# ── Table-aware 분할 유틸리티 ──

def _extract_segments(text: str) -> list[tuple]:
    """텍스트를 ('text', 내용) 또는 ('table', 내용) 세그먼트로 분할한다.

    마크다운 테이블 행(| ... |)을 연속 블록으로 묶어 원자적 단위로 취급한다.
    """
    lines = text.split("\n")
    segments = []
    current_lines = []
    current_type = "text"

    for line in lines:
        stripped = line.strip()
        is_table_row = (
            _TABLE_ROW_RE.match(stripped) is not None
            and len(stripped) > 2
        )

        if is_table_row:
            if current_type == "text" and current_lines:
                segments.append(("text", "\n".join(current_lines)))
                current_lines = []
            current_type = "table"
            current_lines.append(line)
        else:
            if current_type == "table" and current_lines:
                segments.append(("table", "\n".join(current_lines)))
                current_lines = []
            current_type = "text"
            current_lines.append(line)

    if current_lines:
        segments.append((current_type, "\n".join(current_lines)))
    return segments


def _split_large_table(table_text: str, max_size: int = CHUNK_SIZE) -> list[str]:
    """큰 마크다운 테이블을 헤더를 유지하며 행 단위로 분할한다.

    각 서브 청크의 맨 위에 원본 헤더+구분선을 반복 삽입하여
    검색 시 표 구조가 온전히 보이도록 한다.
    """
    lines = table_text.strip().split("\n")
    if len(lines) < 3:
        return [table_text]

    header_line = lines[0]
    has_separator = _TABLE_SEP_RE.match(lines[1].strip()) is not None

    if has_separator:
        header_block = header_line + "\n" + lines[1]
        data_rows = lines[2:]
    else:
        header_block = header_line
        data_rows = lines[1:]

    if not data_rows:
        return [table_text]

    header_len = len(header_block) + 1
    chunks = []
    current_rows = []
    current_len = header_len

    for row in data_rows:
        row_len = len(row) + 1
        if current_len + row_len > max_size and current_rows:
            chunks.append(header_block + "\n" + "\n".join(current_rows))
            current_rows = []
            current_len = header_len
        current_rows.append(row)
        current_len += row_len

    if current_rows:
        chunks.append(header_block + "\n" + "\n".join(current_rows))

    return chunks if chunks else [table_text]


def _merge_small_chunks(chunks: list[str], max_size: int = CHUNK_SIZE) -> list[str]:
    """인접한 작은 청크들을 합쳐 최적 크기로 만든다."""
    if not chunks:
        return []
    merged = [chunks[0]]
    for chunk in chunks[1:]:
        combined_len = len(merged[-1]) + len(chunk) + 2
        if combined_len <= max_size:
            merged[-1] = merged[-1] + "\n\n" + chunk
        else:
            merged.append(chunk)
    return merged


def _split_preserving_tables(text: str) -> list[str]:
    """테이블 구조를 보존하며 텍스트를 청크로 분할한다.

    1. 텍스트를 text/table 세그먼트로 분리
    2. text 세그먼트 → RecursiveCharacterTextSplitter로 분할
    3. table 세그먼트 → 통째로 유지 (초과 시 헤더 반복 분할)
    4. 인접 소형 청크를 병합하여 최적 크기 달성
    """
    segments = _extract_segments(text)
    raw_chunks = []

    for seg_type, content in segments:
        stripped = content.strip()
        if not stripped:
            continue

        if seg_type == "table":
            if len(stripped) <= CHUNK_SIZE:
                raw_chunks.append(stripped)
            else:
                raw_chunks.extend(_split_large_table(stripped))
        else:
            text_chunks = text_splitter.split_text(stripped)
            raw_chunks.extend(text_chunks)

    return _merge_small_chunks(raw_chunks)


# ── 컨텍스트 헤더 + 청크 생성 ──

def _build_context_header(metadata: dict) -> str:
    """청크 앞에 붙일 문서 컨텍스트 헤더를 생성한다.

    임베딩 시 각 청크가 어떤 사업/기관의 내용인지 알 수 있어
    검색 정확도가 크게 향상된다.
    """
    parts = []
    if metadata.get("사업명"):
        parts.append(f"[사업명: {metadata['사업명']}]")
    if metadata.get("발주기관"):
        parts.append(f"[발주기관: {metadata['발주기관']}]")
    if not parts:
        return ""
    return " ".join(parts) + "\n"


def create_chunks(cleaned_text: str, metadata: dict) -> list[Document]:
    """Table-aware 청킹: 마크다운 테이블 구조를 보존하며 분할한다.

    - 표 내부의 수치/항목이 잘리지 않도록 테이블을 원자적 단위로 취급
    - 큰 테이블은 헤더를 반복하며 행 단위로 분할
    - 소형 청크는 인접 블록과 병합하여 최적 크기 유지
    """
    if not cleaned_text.strip():
        return []
    raw_chunks = _split_preserving_tables(cleaned_text)
    header = _build_context_header(metadata)
    chunks = [
        Document(
            page_content=header + chunk,
            metadata={**metadata, "chunk_index": idx, "chunk_total": len(raw_chunks)},
        )
        for idx, chunk in enumerate(raw_chunks)
    ]
    return chunks


def create_meta_summary_chunk(metadata: dict) -> Document:
    """CSV 메타데이터로 요약 청크를 생성한다.
    텍스트에 누락된 예산/기한/기관 정보를 검색 가능하게 만든다.
    """
    parts = []
    if metadata.get("사업명"):
        parts.append(f"사업명: {metadata['사업명']}")
    if metadata.get("발주기관"):
        parts.append(f"발주기관: {metadata['발주기관']}")
    if metadata.get("사업예산"):
        parts.append(f"사업예산: {metadata['사업예산']}")
    if metadata.get("입찰방식"):
        parts.append(f"입찰방식: {metadata['입찰방식']}")
    if metadata.get("제출기한"):
        parts.append(f"제출기한: {metadata['제출기한']}")
    if metadata.get("참가자격"):
        parts.append(f"참가자격: {metadata['참가자격']}")
    if metadata.get("사업요약"):
        parts.append(f"사업요약: {metadata['사업요약']}")
    if metadata.get("공고번호"):
        parts.append(f"공고번호: {metadata['공고번호']}")
    if not parts:
        return None
    content = "\n".join(parts)
    meta = {**metadata, "chunk_index": -1, "chunk_total": 0, "is_meta_summary": "true"}
    return Document(page_content=content, metadata=meta)


print(f"Table-aware 청킹 설정: size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}")
print("  테이블 보존: 마크다운 테이블을 원자적 단위로 유지")
print("  대형 테이블: 헤더 반복 + 행 단위 분할")
print("컨텍스트 헤더 부착 활성화: [사업명] [발주기관]")
print("메타 요약 청크 함수 정의 완료")

Table-aware 청킹 설정: size=800, overlap=200
  테이블 보존: 마크다운 테이블을 원자적 단위로 유지
  대형 테이블: 헤더 반복 + 행 단위 분할
컨텍스트 헤더 부착 활성화: [사업명] [발주기관]
메타 요약 청크 함수 정의 완료


In [9]:
# --- 통합 파싱 파이프라인 (안정성 강화) ---


def _compute_file_hash(file_path: Path) -> str:
    """파일의 SHA256 해시를 계산한다. (Incremental Upsert 비교용)"""
    h = hashlib.sha256()
    with open(file_path, "rb") as f:
        for block in iter(lambda: f.read(8192), b""):
            h.update(block)
    return h.hexdigest()


def _resolve_file_path(file_name: str, files_dir: Path) -> Optional[Path]:
    """파일명으로 실제 경로를 찾는다. 정확 매칭 → 유사 매칭 순."""
    file_path = files_dir / file_name
    if file_path.exists():
        return file_path
    # 파일명이 정확히 일치하지 않으면 유사 파일 검색
    stem = Path(file_name).stem.rstrip()
    candidates = [f for f in files_dir.iterdir() if f.stem.startswith(stem[:20])]
    if len(candidates) == 1:
        return candidates[0]
    return None


# ── 파싱 전 유효성 검사 ──

def validate_data_files(df: pd.DataFrame, files_dir: Path) -> dict:
    """파싱 시작 전 원본 데이터 폴더의 파일 존재 여부를 검증한다.

    Returns:
        dict with keys: total, found, missing, missing_files, found_files
    """
    if not files_dir.exists():
        print(f"[FATAL] 파일 디렉토리 없음: {files_dir}")
        return {"total": len(df), "found": 0, "missing": len(df),
                "missing_files": list(df["파일명"]), "found_files": []}

    missing_files = []
    found_files = []
    for _, row in df.iterrows():
        file_name = str(row["파일명"]).strip()
        resolved = _resolve_file_path(file_name, files_dir)
        if resolved is not None:
            found_files.append(file_name)
        else:
            missing_files.append(file_name)

    result = {
        "total": len(df),
        "found": len(found_files),
        "missing": len(missing_files),
        "missing_files": missing_files,
        "found_files": found_files,
    }

    print(f"=== 파일 유효성 검사 ===")
    print(f"  전체: {result['total']}건")
    print(f"  존재: {result['found']}건")
    print(f"  누락: {result['missing']}건")
    if missing_files:
        for mf in missing_files[:10]:
            print(f"    - {mf}")
        if len(missing_files) > 10:
            print(f"    ... 외 {len(missing_files) - 10}건")
    return result


# ── 에러 로그 저장 ──

def _save_error_log(error_records: list, path: str = ERROR_LOG_PATH):
    """파싱 에러 로그를 JSON으로 저장한다.

    Args:
        error_records: [{"file": str, "error": str, "type": str, "traceback": str}, ...]
    """
    log_data = {
        "generated_at": datetime.now().isoformat(),
        "total_errors": len(error_records),
        "errors": error_records,
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(log_data, f, ensure_ascii=False, indent=2)
    print(f"에러 로그 저장: {path} ({len(error_records)}건)")


# ── 파일 해시 캐시 ──

def _load_hash_cache(path: str = HASH_CACHE_PATH) -> dict:
    """파일 해시 캐시를 로드한다. 없으면 빈 dict 반환."""
    if not Path(path).exists():
        return {}
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except (json.JSONDecodeError, OSError):
        return {}


def _save_hash_cache(hashes: dict, path: str = HASH_CACHE_PATH):
    """파일 해시 캐시를 저장한다."""
    payload = {
        "updated_at": datetime.now().isoformat(),
        "files": hashes,
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


def _compute_all_hashes(df: pd.DataFrame, files_dir: Path) -> dict:
    """전체 파일의 해시를 계산한다. {파일명: 해시값}"""
    hashes = {}
    for _, row in df.iterrows():
        file_name = str(row["파일명"]).strip()
        resolved = _resolve_file_path(file_name, files_dir)
        if resolved is not None:
            hashes[file_name] = _compute_file_hash(resolved)
    return hashes


def _classify_files_by_hash(current_hashes: dict, cached_data: dict) -> dict:
    """현재 해시와 캐시를 비교하여 변경/신규/삭제/미변경 파일을 분류한다."""
    cached_files = cached_data.get("files", {})

    changed = []
    new_files = []
    unchanged = []

    for fname, fhash in current_hashes.items():
        if fname not in cached_files:
            new_files.append(fname)
        elif cached_files[fname] != fhash:
            changed.append(fname)
        else:
            unchanged.append(fname)

    deleted = [f for f in cached_files if f not in current_hashes]

    return {
        "changed": changed,
        "new": new_files,
        "deleted": deleted,
        "unchanged": unchanged,
    }


# ── 파싱 핵심 로직 ──

def _extract_text_from_file(file_path: Path) -> str:
    """파일 확장자에 따라 적절한 파서로 텍스트를 추출한다."""
    suffix = file_path.suffix.lower()
    if suffix == ".pdf":
        return extract_text_from_pdf(str(file_path))
    elif suffix == ".hwpx":
        return extract_text_from_hwpx(str(file_path))
    elif suffix == ".hwp":
        return extract_text_from_hwp(str(file_path))
    elif suffix in (".docx", ".doc"):
        return extract_text_from_hwpx(str(file_path))
    return ""


def _make_chunk_id(source: str, chunk_index: int) -> str:
    """파일명 + 청크 인덱스 기반 결정적 ID를 생성한다."""
    raw = f"{source}::chunk::{chunk_index}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:16]


_ERROR_TAGS = ["[HWP 파싱 오류:", "[HWPX 파싱 오류:", "[PDF 파싱 오류:"]


def parse_single_document(row: pd.Series, files_dir: Path) -> dict:
    """단일 RFP 문서를 안전하게 파싱한다.

    모든 단계에 try-except를 적용하여 개별 파일 오류가
    전체 파이프라인을 중단시키지 않는다.
    """
    file_name = str(row["파일명"]).strip()
    errors = []
    error_records = []

    # Step 1: 파일 탐색 + 텍스트 추출
    raw_text = ""
    file_path = _resolve_file_path(file_name, files_dir)

    if file_path is not None:
        try:
            raw_text = _extract_text_from_file(file_path)
            if any(raw_text.startswith(tag) for tag in _ERROR_TAGS):
                errors.append(raw_text)
                error_records.append({
                    "file": file_name,
                    "error": raw_text,
                    "type": "parse_error",
                    "traceback": "",
                })
                raw_text = ""
        except Exception as e:
            tb = traceback.format_exc()
            err_msg = f"파싱 예외 ({file_name}): {type(e).__name__}: {e}"
            errors.append(err_msg)
            error_records.append({
                "file": file_name,
                "error": err_msg,
                "type": type(e).__name__,
                "traceback": tb,
            })
            raw_text = ""
    else:
        err_msg = f"파일 없음: {file_name}"
        errors.append(err_msg)
        error_records.append({
            "file": file_name,
            "error": err_msg,
            "type": "file_not_found",
            "traceback": "",
        })

    # 파일 파싱 실패 시 CSV 텍스트를 폴백으로 사용
    if not raw_text.strip():
        try:
            csv_text = row.get("텍스트")
            if pd.notna(csv_text) and str(csv_text).strip():
                raw_text = str(csv_text)
                errors.append(f"파일 파싱 실패, CSV 텍스트 폴백 사용: {file_name}")
        except Exception:
            pass

    # Step 2: 정제
    try:
        cleaned_text = clean_text(raw_text)
    except Exception as e:
        tb = traceback.format_exc()
        err_msg = f"텍스트 정제 오류 ({file_name}): {type(e).__name__}: {e}"
        errors.append(err_msg)
        error_records.append({
            "file": file_name, "error": err_msg,
            "type": "clean_error", "traceback": tb,
        })
        cleaned_text = ""

    if not cleaned_text:
        errors.append(f"텍스트 추출 실패: {file_name}")

    # Step 3: 필드 추출 (CSV 메타데이터 + 텍스트 분석)
    try:
        metadata = extract_fields(row, cleaned_text)
    except Exception as e:
        tb = traceback.format_exc()
        err_msg = f"필드 추출 오류 ({file_name}): {type(e).__name__}: {e}"
        errors.append(err_msg)
        error_records.append({
            "file": file_name, "error": err_msg,
            "type": "field_extraction_error", "traceback": tb,
        })
        metadata = {
            "사업명": str(row.get("사업명", "")),
            "발주기관": str(row.get("발주 기관", "")),
            "사업예산": "", "입찰방식": "", "제출기한": "", "참가자격": "",
        }

    metadata["source"] = file_name
    metadata["공고번호"] = str(row.get("공고 번호", ""))
    metadata["공개일자"] = str(row.get("공개 일자", ""))
    metadata["사업요약"] = str(row.get("사업 요약", ""))[:200]

    # Step 4: 청킹 (결정적 ID 부여)
    try:
        chunks = create_chunks(cleaned_text, metadata)
        for chunk in chunks:
            chunk.metadata["id"] = _make_chunk_id(
                file_name, chunk.metadata["chunk_index"]
            )
    except Exception as e:
        tb = traceback.format_exc()
        err_msg = f"청킹 오류 ({file_name}): {type(e).__name__}: {e}"
        errors.append(err_msg)
        error_records.append({
            "file": file_name, "error": err_msg,
            "type": "chunking_error", "traceback": tb,
        })
        chunks = []

    # Step 5: 메타 요약 청크 추가
    try:
        meta_chunk = create_meta_summary_chunk(metadata)
        if meta_chunk:
            meta_chunk.metadata["id"] = _make_chunk_id(file_name, -1)
            chunks.append(meta_chunk)
    except Exception:
        pass  # 메타 요약 실패는 치명적이지 않음

    if not chunks and cleaned_text:
        errors.append(f"청킹 결과 없음: {file_name}")

    return {
        "file_name": file_name,
        "chunks": chunks,
        "metadata": metadata,
        "errors": errors,
        "error_records": error_records,
    }


def parse_all_documents(df: pd.DataFrame, files_dir: Path):
    """전체 RFP 문서를 배치 파싱한다. 개별 파일 오류 시에도 계속 진행."""
    all_chunks = []
    all_metadata = []
    all_errors = []
    all_error_records = []
    skipped = 0
    file_parsed = 0
    csv_fallback = 0

    for idx, row in df.iterrows():
        try:
            result = parse_single_document(row, files_dir)
        except Exception as e:
            # parse_single_document 자체가 크래시 (최후의 보루)
            file_name = str(row.get("파일명", f"row_{idx}")).strip()
            tb = traceback.format_exc()
            err_msg = f"문서 파싱 치명적 오류 ({file_name}): {type(e).__name__}: {e}"
            all_errors.append(err_msg)
            all_error_records.append({
                "file": file_name, "error": err_msg,
                "type": "fatal_parse_error", "traceback": tb,
            })
            skipped += 1
            continue

        all_chunks.extend(result["chunks"])
        all_metadata.append(result["metadata"])
        all_errors.extend(result["errors"])
        all_error_records.extend(result.get("error_records", []))

        if not result["chunks"]:
            skipped += 1
        has_fallback = any("CSV 텍스트 폴백" in e for e in result["errors"])
        if has_fallback:
            csv_fallback += 1
        else:
            file_parsed += 1

        if (idx + 1) % 20 == 0:
            print(f"  파싱: {idx + 1}/{len(df)} 문서")

    print(f"  파일 직접 파싱: {file_parsed}건")
    print(f"  CSV 텍스트 폴백: {csv_fallback}건")
    if skipped:
        print(f"  [주의] 청크 0개 문서: {skipped}건")

    return all_chunks, all_metadata, all_errors, all_error_records


print("파싱 파이프라인 정의 완료 (안전 파싱 + 에러 로그 + 해시 캐시 + 유효성 검사)")

파싱 파이프라인 정의 완료 (안전 파싱 + 에러 로그 + 해시 캐시 + 유효성 검사)


In [10]:
# === 전체 파싱 실행 (유효성 검사 + 에러 로그 + 해시 캐시) ===

# Step 1: 파싱 전 파일 유효성 검사
validation = validate_data_files(df_raw, FILES_DIR)

if validation["found"] == 0:
    raise RuntimeError(
        f"파싱 가능한 파일이 없습니다. 디렉토리를 확인하세요: {FILES_DIR}"
    )

# Step 2: 파일 해시 계산 + 변경 분류
print("\n=== 파일 해시 계산 ===")
current_hashes = _compute_all_hashes(df_raw, FILES_DIR)
cached_hash_data = _load_hash_cache()
file_diff = _classify_files_by_hash(current_hashes, cached_hash_data)

print(f"  신규: {len(file_diff['new'])}건")
print(f"  변경: {len(file_diff['changed'])}건")
print(f"  미변경: {len(file_diff['unchanged'])}건")
print(f"  삭제: {len(file_diff['deleted'])}건")

# Step 3: 전체 파싱 실행
print(f"\n=== RFP 문서 파싱 시작 ===")
print(f"대상 문서 수: {len(df_raw)}\n")

all_chunks, all_metadata, all_errors, all_error_records = parse_all_documents(
    df_raw, FILES_DIR
)

# Step 4: 에러 로그 저장 (에러가 있을 때만)
if all_error_records:
    _save_error_log(all_error_records)
else:
    print("파싱 에러 없음 — error_log.json 생성 건너뜀")

# Step 5: 해시 캐시 저장 (다음 실행에서 Incremental Upsert에 사용)
_save_hash_cache(current_hashes)

# Step 6: 결과 출력
print(f"\n=== 파싱 결과 ===")
print(f"총 청크 수: {len(all_chunks)}")
print(f"문서당 평균: {len(all_chunks) / max(len(df_raw), 1):.1f}개")
print(f"에러 수: {len(all_errors)} (상세: {len(all_error_records)}건 error_log.json)")
if all_errors:
    for err in all_errors[:10]:
        print(f"  - {err}")
    if len(all_errors) > 10:
        print(f"  ... 외 {len(all_errors) - 10}건")

=== 파일 유효성 검사 ===
  전체: 100건
  존재: 100건
  누락: 0건

=== 파일 해시 계산 ===
  신규: 100건
  변경: 0건
  미변경: 0건
  삭제: 0건

=== RFP 문서 파싱 시작 ===
대상 문서 수: 100

  파싱: 20/100 문서
MuPDF error: syntax error: invalid key in dict

MuPDF error: syntax error: invalid key in dict

MuPDF error: syntax error: invalid key in dict

MuPDF error: syntax error: invalid key in dict

MuPDF error: syntax error: invalid key in dict

MuPDF error: syntax error: invalid key in dict

  파싱: 40/100 문서
  파싱: 60/100 문서
  파싱: 80/100 문서
  파싱: 100/100 문서
  파일 직접 파싱: 100건
  CSV 텍스트 폴백: 0건
파싱 에러 없음 — error_log.json 생성 건너뜀

=== 파싱 결과 ===
총 청크 수: 9669
문서당 평균: 96.7개
에러 수: 0 (상세: 0건 error_log.json)


In [11]:
# 파싱 결과 검증
chunk_lengths = [len(c.page_content) for c in all_chunks]
df_meta = pd.DataFrame(all_metadata)

print("=== 청크 길이 통계 ===")
if chunk_lengths:
    print(f"  총: {len(chunk_lengths)}, 최소: {min(chunk_lengths)}, 최대: {max(chunk_lengths)}")
    print(f"  평균: {sum(chunk_lengths)/len(chunk_lengths):.0f}")

print("\n=== 필드 추출 현황 ===")
for col in ["사업명", "발주기관", "사업예산", "입찰방식", "제출기한", "참가자격"]:
    n = df_meta[col].apply(lambda x: bool(x and str(x).strip())).sum()
    print(f"  {col}: {n}/{len(df_meta)} ({n/len(df_meta)*100:.0f}%)")

=== 청크 길이 통계 ===
  총: 9669, 최소: 66, 최대: 4026
  평균: 727

=== 필드 추출 현황 ===
  사업명: 100/100 (100%)
  발주기관: 100/100 (100%)
  사업예산: 99/100 (99%)
  입찰방식: 97/100 (97%)
  제출기한: 92/100 (92%)
  참가자격: 99/100 (99%)


In [12]:
# === 파싱 결과 육안 검증 ===
# HWP 1개 + PDF 1개를 골라서 [원본 → 정제 → 청크] 과정을 단계별로 확인한다.

def inspect_document(file_name: str, show_chars: int = 1500):
    """특정 파일의 파싱 과정을 단계별로 출력한다."""
    row = df_raw[df_raw["파일명"] == file_name]
    if row.empty:
        # 부분 매칭
        row = df_raw[df_raw["파일명"].str.contains(file_name[:20], na=False)]
    if row.empty:
        print(f"파일을 찾을 수 없습니다: {file_name}")
        return
    row = row.iloc[0]
    actual_name = row["파일명"]
    file_path = FILES_DIR / actual_name

    # ── 1단계: 원본 텍스트 추출 ──
    raw_text = _extract_text_from_file(file_path) if file_path.exists() else "(파일 없음)"
    raw_len = len(raw_text)

    # ── 2단계: 정제 (목차 제거 + 부록 제거 + 페이지번호 제거) ──
    cleaned = clean_text(raw_text)
    cleaned_len = len(cleaned)

    # ── 3단계: 해당 파일의 청크 가져오기 ──
    file_chunks = [c for c in all_chunks if c.metadata.get("source") == actual_name]
    meta_chunk = [c for c in file_chunks if c.metadata.get("is_meta_summary") == "true"]
    body_chunks = [c for c in file_chunks if c.metadata.get("is_meta_summary") != "true"]

    # ── 출력 ──
    ext = Path(actual_name).suffix.upper()
    print(f"{'='*70}")
    print(f"  파일: {actual_name} ({ext})")
    print(f"{'='*70}")

    print(f"\n📊 통계:")
    print(f"  원본 텍스트: {raw_len:,}자")
    print(f"  정제 후:     {cleaned_len:,}자 ({(raw_len - cleaned_len):,}자 제거, {(raw_len - cleaned_len) / max(raw_len,1) * 100:.1f}%)")
    print(f"  청크 수:     {len(body_chunks)}개 (+ 메타요약 {len(meta_chunk)}개)")

    if meta_chunk:
        print(f"\n📋 메타 요약 청크:")
        print(f"  {meta_chunk[0].page_content}")

    print(f"\n📄 원본 텍스트 (앞 {show_chars}자):")
    print(f"{'─'*50}")
    print(raw_text[:show_chars])

    print(f"\n✅ 정제 후 텍스트 (앞 {show_chars}자):")
    print(f"{'─'*50}")
    print(cleaned[:show_chars])

    print(f"\n🔖 청크 샘플 (처음 3개):")
    print(f"{'─'*50}")
    for i, chunk in enumerate(body_chunks[:3]):
        idx = chunk.metadata.get("chunk_index", "?")
        print(f"\n  [청크 {idx}] ({len(chunk.page_content)}자)")
        print(f"  {chunk.page_content[:250]}...")

    print(f"\n🔖 청크 샘플 (중간 1개):")
    mid = len(body_chunks) // 2
    if mid < len(body_chunks):
        chunk = body_chunks[mid]
        print(f"  [청크 {chunk.metadata.get('chunk_index', '?')}] ({len(chunk.page_content)}자)")
        print(f"  {chunk.page_content[:250]}...")

    print(f"\n🔖 마지막 청크:")
    if body_chunks:
        chunk = body_chunks[-1]
        print(f"  [청크 {chunk.metadata.get('chunk_index', '?')}] ({len(chunk.page_content)}자)")
        print(f"  {chunk.page_content[:250]}...")
    print()


# ── HWP 파일 검증 ──
hwp_files = df_raw[df_raw["파일형식"] == "hwp"]["파일명"].tolist()
print("▶ HWP 파일 파싱 검증")
inspect_document(hwp_files[0])

# ── PDF 파일 검증 ──
pdf_files = df_raw[df_raw["파일형식"] == "pdf"]["파일명"].tolist()
print("\n▶ PDF 파일 파싱 검증")
inspect_document(pdf_files[0])


▶ HWP 파일 파싱 검증
  파일: 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp (.HWP)

📊 통계:
  원본 텍스트: 49,620자
  정제 후:     22,898자 (26,722자 제거, 53.9%)
  청크 수:     40개 (+ 메타요약 1개)

📋 메타 요약 청크:
  사업명: 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화
발주기관: 한영대학
사업예산: 1.3억원
입찰방식: 협상에 의한 계약
제출기한: 2024-10-15 17:00:00
참가자격: ○ 본 제안 사업의 수행이 가능한 업체로서 다음 요건을 모두 갖춘 사업자
① 「국가를 당사자로 하는 계약에 관한법률 시행령」 제12조 및 동법 시행규칙 제14조 규정에 의한 경쟁입찰 참가자격을 갖춘 업체
② 「소프트웨어산업진흥법」  제24조 및 동법 시행령 제17조에 의한 소프트웨어사업자 (컴퓨터관련서비스업) 신고를 필한 업체
③ 중소벤처기업부 고시 「중소기업자간 경쟁제품 직접생산확인기준」에 따라 직접생산증명서 [입찰마감일 전일까지 소프트웨어엔지니어링업 정보시스템개발서비스(세부품명번호: 8111159901)로 발행된 것으로 유효기간 내에 있어야 함]를 소지한 업체
④ 최근 3년 이내에 ASP.NET으로 대학관련 종합정보/학생이력/역량시스템 구축 개발 수주가 단일건으로 1억원 이상 실적이 있는 업체
 Windows Server, MS-SQL기반 ASP.NET 개발환경에 구축 실적이 있는 업체
➅ .NET기반의 Windows Server의 Application 생성 및 표준화된 표준프레임워크을보유하고 있는 업체(타대학교에 적용 및 검증된 프레임워크 활용)
➆ 공동수급(공동이행방식) 및 하도급은 불가함의 규정에 의한 경쟁 입찰 참가 자격을 갖춘 과업 수행 가능자
3) 입찰 계약조건
○ 계약의 체결 및 이행에 관하여는 「협상에 의한 계약체결 기준」 제14조의 규정에 의하여 서면 통보한 협상 결과와 「국가를 당사자로 하는 계약에 관한 법률」 및 동

In [13]:
# 사업예산이 비어 있는 파일들만 골라내기
missing_budget = df_meta[df_meta['사업예산'] == ""]['source'].tolist()
print(f"누락 건수: ({len(missing_budget)}건): {missing_budget}")

# 제출기한이 비어 있는 파일들만 골라보기
missing_deadline = df_meta[df_meta['제출기한'] == ""]['source'].tolist()
print(f"제출기한 누락 ({len(missing_deadline)}건): {missing_deadline}")

# 입찰방식이 비어 있는 파일들만 골라보기
missing_bid_type = df_meta[df_meta['입찰방식'] == ""]['source'].tolist()
print(f"입찰방식 누락 ({len(missing_bid_type)}건): {missing_bid_type}")

# 참가자격이 비어 있는 파일들만 골라보기
missing_qual = df_meta[df_meta['참가자격'] == ""]['source'].tolist()
print(f"참가자격 누락 ({len(missing_qual)}건): {missing_qual}")

누락 건수: (1건): ['대한상공회의소_기업 재생에너지 지원센터 홈페이지 개편 및 시스템 고.hwp']
제출기한 누락 (8건): ['서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf', '경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp', '한국수자원공사_건설통합시스템(CMS) 고도화.hwp', '한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp', 'KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp', 'BioIN_의료기기산업 종합정보시스템(정보관리기관) 기능개선 사업(2차).hwp', '국가철도공단_철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역(변.hwp', '세종테크노파크_세종테크노파크 인사정보 전산시스템 구축 용역 입찰공.hwp']
입찰방식 누락 (3건): ['한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp', '한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp', '기초과학연구원_2025년도 중이온가속기용 극저온시스템 운전 용역.pdf']
참가자격 누락 (1건): ['기초과학연구원_2025년도 중이온가속기용 극저온시스템 운전 용역.pdf']


In [14]:
target_file = '한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp'
target_meta = next((m for m in all_metadata if m['source'] == target_file), None)

if target_meta:
    print(f"--- '{target_file}'의 개선된 결과 ---")
    print(f"📝 참가자격: {target_meta.get('참가자격')[:100]}...")

--- '한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp'의 개선된 결과 ---
📝 참가자격: 1.12 대내외 인․허가 및 심의자료의 제출
계약상대자는 이 사업추진을 위해 과업중 또는 완료후 각종 관련법규에 따라 사전 인․허가 및 심의 등이 필요한 사항들을 파악하여야 하며 ...


---
## 3. Retriever — 문서 검색

Parser의 청크를 벡터 DB에 적재하고, 유사도 기반으로 검색한다.
```
[all_chunks] → [Embedding] → [Chroma 적재] → [유사도 검색] → [GraphState 반환]
```

In [15]:
# --- 검색 설정 ---
EMBEDDING_MODEL = "text-embedding-3-small"
COLLECTION_NAME = "bidmate_rfp"
DEFAULT_K = 5
DEFAULT_SEARCH_TYPE = "hybrid"   # "similarity" | "mmr" | "hybrid"
SCORE_THRESHOLD = 0.2
BM25_WEIGHT = 0.3                # 하이브리드 검색에서 BM25 가중치 (0~1)
SEMANTIC_WEIGHT = 0.7            # 하이브리드 검색에서 시맨틱 가중치

# --- Cross-Encoder Re-ranker 설정 ---
RERANK_ENABLED = FLASHRANK_AVAILABLE   # flashrank 설치 시 자동 활성화
RERANK_MODEL = "ms-marco-MultiBERT-L-12"  # 다국어 Cross-Encoder (한국어 지원)
RERANK_FETCH_K = 10              # 재정렬 전 하이브리드 검색 후보 수
RERANK_TOP_N = 5                 # 재정렬 후 최종 반환 개수


def create_embeddings(model_name: str = EMBEDDING_MODEL) -> OpenAIEmbeddings:
    """임베딩 모델을 생성한다. Vertex AI 교체 시 이 함수만 수정."""
    return OpenAIEmbeddings(model=model_name)


embeddings = create_embeddings()

# 동작 확인
test_vec = embeddings.embed_query("테스트")
print(f"임베딩 모델: {EMBEDDING_MODEL}")
print(f"임베딩 차원: {len(test_vec)}")
print(f"검색 전략: {DEFAULT_SEARCH_TYPE} (BM25 {BM25_WEIGHT} + Semantic {SEMANTIC_WEIGHT})")
print(f"유사도 임계값: {SCORE_THRESHOLD}")
print(f"Re-ranker: {'ON' if RERANK_ENABLED else 'OFF'}", end="")
if RERANK_ENABLED:
    print(f" (model={RERANK_MODEL}, fetch={RERANK_FETCH_K} → top={RERANK_TOP_N})")
else:
    print()

임베딩 모델: text-embedding-3-small
임베딩 차원: 1536
검색 전략: hybrid (BM25 0.3 + Semantic 0.7)
유사도 임계값: 0.2
Re-ranker: OFF


In [16]:
# --- 벡터스토어 생성/로드 (Incremental Upsert) ---
import chromadb
import shutil


def create_vectorstore(documents, embedding_fn, collection_name=COLLECTION_NAME,
                       persist_dir=PERSIST_DIR):
    """청크를 Chroma에 적재한다. 결정적 ID로 중복 삽입을 방지."""
    if not documents:
        raise ValueError("적재할 Document가 없습니다. Parser 결과를 확인하세요.")
    ids = [doc.metadata.get("id", str(i)) for i, doc in enumerate(documents)]
    return Chroma.from_documents(
        documents=documents,
        embedding=embedding_fn,
        collection_name=collection_name,
        persist_directory=persist_dir,
        ids=ids,
    )


def load_vectorstore(embedding_fn, collection_name=COLLECTION_NAME,
                     persist_dir=PERSIST_DIR):
    """영속화된 Chroma를 로드한다. 손상 시 None 반환."""
    if not Path(persist_dir).exists():
        return None
    try:
        store = Chroma(
            collection_name=collection_name,
            embedding_function=embedding_fn,
            persist_directory=persist_dir,
        )
        store._collection.count()
        return store
    except Exception as e:
        print(f"DB 로드 실패 (손상 가능): {e}")
        return None


def _force_remove_db(persist_dir):
    """DB 디렉토리를 강제 삭제한다. 실패 시 내부 파일만 정리."""
    db_path = Path(persist_dir)
    if not db_path.exists():
        return
    try:
        shutil.rmtree(db_path)
        print(f"DB 디렉토리 삭제 완료: {persist_dir}")
    except PermissionError:
        for item in list(db_path.iterdir()):
            if item.is_dir():
                shutil.rmtree(item, ignore_errors=True)
        print(f"DB 세그먼트 정리 완료 (sqlite 잠금 우회)")
    except Exception as e:
        print(f"DB 삭제 실패: {e}")


def reset_collection(collection_name=COLLECTION_NAME, persist_dir=PERSIST_DIR):
    """ChromaDB API로 컬렉션을 삭제한다."""
    try:
        client = chromadb.PersistentClient(path=persist_dir)
        names = [c.name for c in client.list_collections()]
        if collection_name in names:
            client.delete_collection(name=collection_name)
            print(f"컬렉션 '{collection_name}' 삭제 완료")
        del client
    except Exception as e:
        print(f"컬렉션 API 삭제 실패: {e}")
        _force_remove_db(persist_dir)


def _delete_chunks_by_source(collection, source_file: str) -> int:
    """특정 파일의 청크를 Chroma에서 삭제한다. 삭제된 수 반환."""
    try:
        before = collection.count()
        collection.delete(where={"source": source_file})
        after = collection.count()
        return before - after
    except Exception:
        return 0


def _add_chunks_batch(store, chunks: list, batch_size: int = 500):
    """청크를 배치 단위로 Chroma에 추가한다."""
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]
        ids = [doc.metadata.get("id", str(i + j)) for j, doc in enumerate(batch)]
        texts = [doc.page_content for doc in batch]
        metadatas = [doc.metadata for doc in batch]
        store._collection.upsert(
            ids=ids,
            documents=texts,
            metadatas=metadatas,
            embeddings=store._embedding_function.embed_documents(texts),
        )


def incremental_upsert(all_chunks, embedding_fn, file_diff,
                       collection_name=COLLECTION_NAME, persist_dir=PERSIST_DIR):
    """파일 해시 비교 기반 Incremental Upsert.

    - 미변경 파일: 건너뜀 (임베딩 API 비용 절약)
    - 변경/신규 파일: 기존 청크 삭제 → 새 청크 삽입
    - 삭제 파일: 기존 청크 삭제

    Args:
        all_chunks: 전체 Document 리스트
        embedding_fn: 임베딩 함수
        file_diff: _classify_files_by_hash() 결과
        collection_name: Chroma 컬렉션명
        persist_dir: Chroma 영속화 경로

    Returns:
        Chroma vectorstore 인스턴스
    """
    files_to_update = set(file_diff["changed"] + file_diff["new"])
    files_to_delete = set(file_diff["deleted"])
    unchanged_count = len(file_diff["unchanged"])

    store = load_vectorstore(embedding_fn, collection_name, persist_dir)

    # DB가 없으면 전체 새로 생성
    if store is None:
        _force_remove_db(persist_dir)
        assert len(all_chunks) > 0, "all_chunks가 비어있습니다."
        print(f"새 벡터스토어 생성 중... ({len(all_chunks)}개 청크)")
        store = create_vectorstore(all_chunks, embedding_fn, collection_name, persist_dir)
        print(f"생성 완료: {store._collection.count()}개 청크")
        return store

    # 변경/삭제 사항이 없으면 기존 DB 그대로 사용
    if not files_to_update and not files_to_delete:
        print(f"변경 사항 없음 — 기존 벡터스토어 유지 ({store._collection.count()}개 청크)")
        return store

    collection = store._collection
    deleted_total = 0
    added_total = 0

    # 삭제 파일의 청크 제거
    for fname in files_to_delete:
        n = _delete_chunks_by_source(collection, fname)
        deleted_total += n
        if n > 0:
            print(f"  [삭제] {fname}: {n}개 청크 제거")

    # 변경/신규 파일: 기존 청크 삭제 → 새 청크 삽입
    chunks_to_add = [
        c for c in all_chunks
        if c.metadata.get("source") in files_to_update
    ]

    for fname in files_to_update:
        n = _delete_chunks_by_source(collection, fname)
        deleted_total += n

    if chunks_to_add:
        print(f"  변경/신규 청크 삽입 중... ({len(chunks_to_add)}개)")
        _add_chunks_batch(store, chunks_to_add)
        added_total = len(chunks_to_add)

    final_count = collection.count()
    print(f"\n=== Incremental Upsert 완료 ===")
    print(f"  미변경: {unchanged_count}건 (건너뜀)")
    print(f"  삭제: {deleted_total}개 청크 제거")
    print(f"  추가: {added_total}개 청크 삽입")
    print(f"  최종: {final_count}개 청크")

    return store


# === 메인 로직: Incremental Upsert 적용 ===
has_changes = (
    len(file_diff["changed"]) > 0
    or len(file_diff["new"]) > 0
    or len(file_diff["deleted"]) > 0
)

if has_changes:
    print("파일 변경 감지 → Incremental Upsert 수행")
    vectorstore = incremental_upsert(all_chunks, embeddings, file_diff)
else:
    # 변경 없지만 DB가 없을 수도 있으므로 확인
    existing = load_vectorstore(embeddings)
    if existing is not None:
        vectorstore = existing
        print(f"기존 벡터스토어 로드: {vectorstore._collection.count()}개 청크")
    else:
        print("DB 없음 → 전체 생성")
        _force_remove_db(PERSIST_DIR)
        vectorstore = create_vectorstore(all_chunks, embeddings)
        print(f"생성 완료: {vectorstore._collection.count()}개 청크")

print(f"영속화 경로: {PERSIST_DIR}")

파일 변경 감지 → Incremental Upsert 수행
DB 로드 실패 (손상 가능): Error executing plan: Error sending backfill request to compactor: Error constructing hnsw segment reader: Error creating hnsw segment reader: Error loading hnsw index
DB 세그먼트 정리 완료 (sqlite 잠금 우회)
새 벡터스토어 생성 중... (9669개 청크)
생성 완료: 10370개 청크
영속화 경로: c:\Users\User\Desktop\중급 프로젝트\chroma_db_v2


In [17]:
# --- 검색 함수 (BM25 하이브리드, Pickle 캐시 + Re-ranker + 비동기) ---
import asyncio

MAX_CONTEXT_CHARS = 4000

# ── Kiwi 한국어 토크나이저 초기화 ──
_kiwi = Kiwi()


def _tokenize_ko(text: str) -> list[str]:
    """Kiwi 형태소 분석기로 한국어 텍스트를 토큰화한다."""
    tokens = _kiwi.tokenize(text)
    return [t.form for t in tokens if len(t.form) > 1 or t.tag.startswith("N")]


# ── Cross-Encoder Re-ranker 초기화 ──
_reranker = None
if RERANK_ENABLED:
    print(f"Re-ranker 모델 로드 중: {RERANK_MODEL}...")
    _reranker = Ranker(model_name=RERANK_MODEL)
    print("Re-ranker 초기화 완료")


def _rerank_sync(query: str, passages: list[dict]) -> list:
    """동기 re-ranking (CPU-bound, to_thread용)."""
    request = RerankRequest(query=query, passages=passages)
    return _reranker.rerank(request)


async def rerank_documents(
    query: str,
    scored_docs: list[tuple],
    top_n: int = RERANK_TOP_N,
) -> list[tuple]:
    """Cross-Encoder로 검색 결과를 재정렬한다.

    하이브리드 검색(BM25+시맨틱)의 상위 후보를 Cross-Encoder가
    query-document 쌍 단위로 정밀 평가하여 최종 순위를 결정한다.

    Args:
        query: 사용자 질문
        scored_docs: (Document, score) 튜플 리스트 (하이브리드 검색 결과)
        top_n: 재정렬 후 반환할 최대 문서 수

    Returns:
        재정렬된 (Document, rerank_score) 튜플 리스트
    """
    if not _reranker or not scored_docs:
        return scored_docs[:top_n]

    # FlashRank 입력 형식으로 변환
    passages = [
        {"id": i, "text": doc.page_content[:1000]}
        for i, (doc, _) in enumerate(scored_docs)
    ]

    # CPU-bound → 별도 스레드에서 실행
    reranked = await asyncio.to_thread(_rerank_sync, query, passages)

    # 재정렬 결과를 원본 Document에 매핑
    result = []
    for item in reranked[:top_n]:
        orig_idx = item["id"]
        orig_doc = scored_docs[orig_idx][0]
        result.append((orig_doc, float(item["score"])))

    return result


# ── BM25 인덱스 Pickle 저장/로드 ──

def _build_bm25_index():
    """벡터스토어에서 BM25 인덱스를 새로 구축한다."""
    data = vectorstore._collection.get(include=["documents", "metadatas"])
    docs = list(data["documents"])
    metas = list(data["metadatas"])
    corpus = [_tokenize_ko(doc) for doc in docs]
    index = BM25Okapi(corpus)
    return docs, metas, corpus, index


def _save_bm25_index(docs, metas, corpus, index, path=BM25_INDEX_PATH):
    """BM25 인덱스를 Pickle로 로컬에 저장한다."""
    payload = {
        "docs": docs,
        "metas": metas,
        "corpus": corpus,
        "index": index,
        "doc_count": len(docs),
    }
    with open(path, "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)
    size_mb = Path(path).stat().st_size / (1024 * 1024)
    print(f"BM25 인덱스 저장 완료: {path} ({size_mb:.1f} MB)")


def _load_bm25_index(path=BM25_INDEX_PATH):
    """Pickle에서 BM25 인덱스를 로드한다. 문서 수 불일치 시 None 반환."""
    if not Path(path).exists():
        return None
    try:
        with open(path, "rb") as f:
            payload = pickle.load(f)
        expected = vectorstore._collection.count()
        if payload["doc_count"] != expected:
            print(f"BM25 캐시 불일치 (캐시: {payload['doc_count']}, DB: {expected})")
            return None
        return payload
    except Exception as e:
        print(f"BM25 캐시 로드 실패: {e}")
        return None


# === BM25 인덱스 로드 또는 재구축 ===
print("BM25 인덱스 로드 중...")
_start = time.time()
_cached = _load_bm25_index()

if _cached is not None:
    _bm25_docs = _cached["docs"]
    _bm25_metas = _cached["metas"]
    _bm25_index = _cached["index"]
    _elapsed = time.time() - _start
    print(f"BM25 캐시 로드 완료: {len(_bm25_docs)}개 문서 ({_elapsed:.2f}s)")
else:
    print("BM25 인덱스 새로 구축 중...")
    _bm25_docs, _bm25_metas, _bm25_corpus, _bm25_index = _build_bm25_index()
    _save_bm25_index(_bm25_docs, _bm25_metas, _bm25_corpus, _bm25_index)
    _elapsed = time.time() - _start
    print(f"BM25 인덱스 구축+저장 완료: {len(_bm25_docs)}개 문서 ({_elapsed:.2f}s)")

del _cached  # 메모리 해제


# === 비동기 검색 함수 ===

def search_bm25(query: str, k: int = DEFAULT_K) -> list[tuple]:
    """BM25 키워드 검색. (Document, score) 튜플 리스트를 반환한다."""
    tokenized_query = _tokenize_ko(query)
    scores = _bm25_index.get_scores(tokenized_query)
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    results = []
    for idx in top_indices:
        doc = Document(
            page_content=_bm25_docs[idx],
            metadata=_bm25_metas[idx],
        )
        results.append((doc, float(scores[idx])))
    return results


async def search_similarity(query, k=DEFAULT_K, score_threshold=SCORE_THRESHOLD):
    """비동기 시맨틱 유사도 검색. 임계값 미달 시 top-k 폴백."""
    results = await vectorstore.asimilarity_search_with_relevance_scores(query, k=k)
    filtered = [(doc, score) for doc, score in results if score >= score_threshold]
    if not filtered and results:
        return results
    return filtered


async def search_hybrid(query: str, k: int = DEFAULT_K, bm25_weight: float = BM25_WEIGHT,
                        semantic_weight: float = SEMANTIC_WEIGHT) -> list[tuple]:
    """비동기 BM25 + 시맨틱 하이브리드 검색 (RRF 기반 점수 결합).

    BM25(CPU-bound)와 시맨틱(IO-bound) 검색을 병렬 실행 후 RRF로 결합한다.
    """
    fetch_k = k * 3

    # BM25는 CPU-bound → to_thread, 시맨틱은 네이티브 비동기
    semantic_coro = vectorstore.asimilarity_search_with_relevance_scores(query, k=fetch_k)
    bm25_coro = asyncio.to_thread(search_bm25, query, fetch_k)

    semantic_results, bm25_results = await asyncio.gather(semantic_coro, bm25_coro)

    # RRF 점수 계산: score = weight / (rank + 60)
    rrf_scores = {}
    rrf_docs = {}

    for rank, (doc, _score) in enumerate(semantic_results):
        doc_key = doc.metadata.get("id", doc.page_content[:100])
        rrf_scores[doc_key] = rrf_scores.get(doc_key, 0) + semantic_weight / (rank + 60)
        rrf_docs[doc_key] = doc

    for rank, (doc, _score) in enumerate(bm25_results):
        doc_key = doc.metadata.get("id", doc.page_content[:100])
        rrf_scores[doc_key] = rrf_scores.get(doc_key, 0) + bm25_weight / (rank + 60)
        rrf_docs[doc_key] = doc

    sorted_keys = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)[:k]
    return [(rrf_docs[key], rrf_scores[key]) for key in sorted_keys]


async def search_mmr(query, k=DEFAULT_K, fetch_k=20, lambda_mult=0.5):
    """비동기 MMR 검색: 유사도 + 다양성."""
    return await vectorstore.amax_marginal_relevance_search(
        query, k=k, fetch_k=fetch_k, lambda_mult=lambda_mult
    )


async def search_with_filter(query, filters, k=DEFAULT_K):
    """비동기 메타데이터 필터 적용 검색."""
    where = {}
    if filters.get("기관"):
        where["발주기관"] = filters["기관"]
    if filters.get("파일"):
        where["source"] = filters["파일"]
    search_kwargs = {"k": k}
    if where:
        search_kwargs["filter"] = where
    return await vectorstore.asimilarity_search(query, **search_kwargs)


def _truncate_contexts(contexts: list[str], max_chars: int = MAX_CONTEXT_CHARS) -> list[str]:
    """총 문자 수가 max_chars를 넘지 않도록 컨텍스트를 잘라낸다."""
    result, total = [], 0
    for ctx in contexts:
        if total + len(ctx) > max_chars:
            remaining = max_chars - total
            if remaining > 200:
                result.append(ctx[:remaining] + "...")
            break
        result.append(ctx)
        total += len(ctx)
    return result


async def retrieve_for_graph(question, filters=None, search_type=DEFAULT_SEARCH_TYPE, k=DEFAULT_K):
    """비동기 GraphState 호환 통합 검색 함수.

    하이브리드 검색 시 Re-ranker가 활성화되어 있으면:
    1. 상위 RERANK_FETCH_K(10)개 후보를 하이브리드 검색
    2. Cross-Encoder로 재정렬
    3. 상위 RERANK_TOP_N(5)개 최종 반환
    """
    start_ts = time.time()
    error_msg = None
    scores_list = []
    try:
        if filters:
            docs = await search_with_filter(question, filters, k=k)
            scored = [(doc, 0.0) for doc in docs]
        elif search_type == "mmr":
            docs = await search_mmr(question, k=k)
            scored = [(doc, 0.0) for doc in docs]
        elif search_type == "hybrid":
            # Re-ranker 활성화 시: 더 많이 검색 → 재정렬 → 최종 k개
            fetch_k = RERANK_FETCH_K if RERANK_ENABLED else k
            scored = await search_hybrid(question, k=fetch_k)
            if RERANK_ENABLED and len(scored) > 1:
                scored = await rerank_documents(question, scored, top_n=k)
        else:
            scored = await search_similarity(question, k=k)

        docs = [doc for doc, _ in scored]
        scores_list = [s for _, s in scored]
        contexts = [doc.page_content for doc in docs]
        citations = [doc.metadata.get("source", "N/A") for doc in docs]
    except Exception as e:
        error_msg = f"검색 오류: {e}"
        contexts, citations = [], []

    avg_score = sum(scores_list[:3]) / max(len(scores_list[:3]), 1) if scores_list else 0.0
    result = {
        "retrieved_contexts": contexts,
        "citations_used": citations,
        "latency": round(time.time() - start_ts, 4),
        "relevance_score": round(avg_score, 4),
    }
    if error_msg:
        result["errors"] = [error_msg]
    return result


print("비동기 검색 함수 정의 완료: search_similarity, search_mmr, search_hybrid, search_with_filter")
print(f"기본 검색 전략: {DEFAULT_SEARCH_TYPE}")
print(f"BM25 인덱스 경로: {BM25_INDEX_PATH}")
if RERANK_ENABLED:
    print(f"Re-ranker 파이프라인: hybrid(top {RERANK_FETCH_K}) → CrossEncoder → top {RERANK_TOP_N}")

BM25 인덱스 로드 중...
BM25 캐시 불일치 (캐시: 10354, DB: 10370)
BM25 인덱스 새로 구축 중...
BM25 인덱스 저장 완료: c:\Users\User\Desktop\중급 프로젝트\bm25_index.pkl (60.6 MB)
BM25 인덱스 구축+저장 완료: 10370개 문서 (59.52s)
비동기 검색 함수 정의 완료: search_similarity, search_mmr, search_hybrid, search_with_filter
기본 검색 전략: hybrid
BM25 인덱스 경로: c:\Users\User\Desktop\중급 프로젝트\bm25_index.pkl


In [18]:
# Retriever 동작 테스트 (하이브리드 검색, 비동기)
test_queries = [
    "이 사업의 총 예산은 얼마인가요?",
    "제안서 제출 마감일은 언제인가요?",
    "참가 자격 요건은 무엇인가요?",
]

for test_q in test_queries:
    result = await retrieve_for_graph(test_q)
    print(f"\n{'='*60}")
    print(f"질문: {test_q}")
    print(f"검색 전략: {DEFAULT_SEARCH_TYPE} | 결과: {len(result['retrieved_contexts'])}개 | 시간: {result['latency']}s")
    for i, (ctx, src) in enumerate(zip(result["retrieved_contexts"][:3], result["citations_used"][:3])):
        print(f"\n  --- 결과 {i+1} [{src}] ---")
        display = ctx[:300].replace("\n", " ")
        print(f"  {display}...")


질문: 이 사업의 총 예산은 얼마인가요?
검색 전략: hybrid | 결과: 5개 | 시간: 0.1719s

  --- 결과 1 [서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 구축.hwp] ---
  [사업명: 서민금융진흥원 서민금융 채팅 상담시스템 구축] [발주기관: 서민금융진흥원] | 대상 업체 | 사업금액의 하한 | | --- | --- | | 매출액 8천억 원 이상인 대기업 | 80억 원 이상 | | 매출액 8천억 원 미만인 대기업 | 40억 원 이상 |...

  --- 결과 2 [한국재정정보원_e나라도움 업무시스템 웹 접근성 컨설팅.hwp] ---
  [사업명: e나라도움 업무시스템 웹 접근성 컨설팅] [발주기관: 한국재정정보원] | 대상업체 | 사업금액의 하한 | | --- | --- | | 매출액 8천억원 이상인 대기업 | 80억원 이상 | | 매출액 8천억원 미만인 대기업 | 40억원 이상 | | 중견기업 | 20억원 이상 |...

  --- 결과 3 [한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp] ---
  [사업명: 아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아)사업 PMC 용역] [발주기관: 한국농어촌공사] | 고정단가 확인 필수 (단위 : 원) 구 분 | 총사업비 | 용역비 산출기준 | 비 고 | | --- | --- | --- | --- | | 계 | 977,240,000 | | 1USD=1,300원 | | 1. 직접비 | 604,600,760 | | | | 1) 식량안보예측모델 | | | 고정단가, 실비정산 | | - 마스터플랜 수립 | | | | | - 데이터베이스 구축 | | | | | - 시스템 개발 | ...

질문: 제안서 제출 마감일은 언제인가요?
검색 전략: hybrid | 결과: 5개 | 시간: 0.1531s

  --- 결과 1 [한국해양조사협회_2024년 항해용 간행물 품질관리 업무보조 시스템 구축.hwp] ---
  [사업명: 2024년 항해용 간

---
## 4. Prompt — 프롬프트 엔지니어링

3가지 LCEL 체인을 구성한다:
- **Generate**: 검색 컨텍스트 기반 답변 생성
- **Grade**: 문서 관련성 이진 판별
- **Rewrite**: 검색 최적화 질문 재작성

In [19]:
# === 프롬프트 템플릿 ===

GENERATE_SYSTEM = """너는 B2G(정부/공공기관) 입찰 컨설팅 전문 AI 비서 '입찰메이트'야.

[핵심 규칙]
1. 반드시 아래 제공된 RFP 문서 컨텍스트에 기반하여 답변해.
2. 컨텍스트에 없는 내용은 절대 추측하지 말고, "제공된 문서에서 해당 정보를 확인할 수 없습니다."라고 답변해.
3. 답변 말미에 참고한 문서 출처를 반드시 명시해.
4. 금액, 일자, 자격 요건 등 핵심 수치는 원문 그대로 인용해.
5. 여러 문서에 걸쳐 정보가 있으면 통합하여 정리해.

[답변 형식]
- 핵심 내용을 먼저 요약
- 세부 사항은 불릿 포인트로 정리
- 마지막에 [출처] 표기
"""

GENERATE_HUMAN = """다음 RFP 문서 내용을 참고하여 질문에 답변해주세요.

=== 참고 문서 ===
{context}

=== 문서 출처 ===
{citations}

=== 질문 ===
{question}
"""

GRADE_SYSTEM = """너는 RFP(제안요청서) 문서 검수 전문가야.
아래 문서 내용이 질문에 답하는 데 필요한 핵심 정보를 담고 있는지 판별해줘.

[판별 기준]
- 질문의 핵심 주제와 관련된 구체적 정보가 있으면 'yes'
- 질문과 무관하거나 답변에 필요한 정보가 없으면 'no'
- 부분적으로 관련있더라도 핵심 정보가 있으면 'yes'

반드시 'yes' 또는 'no'로만 답변해."""

GRADE_HUMAN = """질문: {question}

문서 내용: {context}"""

REWRITE_SYSTEM = """너는 B2G 입찰 문서 검색 최적화 전문가야.
사용자의 질문을 벡터 검색에 최적화된 형태로 재작성해줘.

[규칙]
1. 30자 이내로 짧게 재작성해. 길면 검색 품질이 떨어져.
2. RFP 도메인 동의어를 1-2개 추가해. (예: 예산→추정가격, 마감→제출기한)
3. 재작성된 질문만 출력해."""

REWRITE_HUMAN = """원래 질문: {question}

짧고 핵심적인 검색 질문으로 재작성해."""


# --- 배치 평가 프롬프트 (5개 문서를 한 번에 평가) ---
BATCH_GRADE_SYSTEM = """너는 RFP(제안요청서) 문서 검수 전문가야.
아래 여러 문서가 질문에 답하는 데 필요한 핵심 정보를 담고 있는지 한꺼번에 판별해줘.

[판별 기준]
- 질문의 핵심 주제와 관련된 구체적 정보가 있으면 'yes'
- 질문과 무관하거나 답변에 필요한 정보가 없으면 'no'
- 부분적으로 관련있더라도 핵심 정보가 있으면 'yes'

각 문서에 대해 순서대로 'yes' 또는 'no'를 판별해.
반드시 문서 개수만큼 정확히 결과를 반환해."""

BATCH_GRADE_HUMAN = """질문: {question}

=== 평가 대상 문서 목록 ===
{numbered_docs}

위 각 문서에 대해 순서대로 관련성을 판별해."""

# (아래에서 통합 print)
# --- Faithfulness 평가 프롬프트 (답변 충실도) ---
FAITHFULNESS_SYSTEM = """너는 RAG 시스템의 답변 충실도(Faithfulness) 평가 전문가야.
생성된 답변이 제공된 컨텍스트(검색 문서)에만 근거하는지 평가해.

[평가 기준]
1. 답변의 모든 주장/사실이 컨텍스트에서 직접 확인 가능하면 높은 점수
2. 컨텍스트에 없는 정보를 추측하거나 만들어냈으면 낮은 점수
3. 숫자, 날짜, 금액 등 핵심 수치가 정확히 인용되었는지 확인

[점수 기준]
- 1.0: 완전 충실 — 모든 내용이 컨텍스트에 근거
- 0.8: 대부분 충실 — 사소한 일반 지식 추가만 있음
- 0.5: 부분 충실 — 일부 내용이 컨텍스트에 없음
- 0.2: 대부분 불충실 — 많은 내용이 근거 없음
- 0.0: 완전 불충실 — 할루시네이션

반드시 점수와 간단한 근거를 제공해."""

FAITHFULNESS_HUMAN = """=== 검색된 컨텍스트 ===
{context}

=== 생성된 답변 ===
{answer}

위 답변이 컨텍스트에만 근거하는지 평가해주세요."""


# --- Answer Relevancy 평가 프롬프트 (질문 적합성) ---
RELEVANCY_SYSTEM = """너는 RAG 시스템의 답변 적합성(Answer Relevancy) 평가 전문가야.
생성된 답변이 사용자의 질문에 정확하고 완전하게 응답하는지 평가해.

[평가 기준]
1. 질문의 핵심 의도를 정확히 파악하고 답변했는지
2. 필요한 핵심 정보(예산, 기한, 자격 등)가 빠짐없이 포함되었는지
3. 불필요한 정보 없이 간결하게 답변했는지

[점수 기준]
- 1.0: 완벽 적합 — 질문에 정확하고 완전하게 답변
- 0.8: 대체로 적합 — 핵심은 답변했으나 일부 세부 정보 누락
- 0.5: 부분 적합 — 관련은 있으나 핵심을 놓침
- 0.2: 거의 부적합 — 질문과 동떨어진 답변
- 0.0: 완전 부적합 — 질문에 전혀 답변하지 않음

반드시 점수와 간단한 근거를 제공해."""

RELEVANCY_HUMAN = """=== 질문 ===
{question}

=== 생성된 답변 ===
{answer}

위 답변이 질문에 적합한지 평가해주세요."""

print("프롬프트 템플릿 정의 완료: GENERATE, GRADE, BATCH_GRADE, REWRITE, FAITHFULNESS, RELEVANCY")

프롬프트 템플릿 정의 완료: GENERATE, GRADE, BATCH_GRADE, REWRITE


In [20]:
# === 출력 스키마 + LLM + LCEL 체인 ===

class GradeAnswer(BaseModel):
    """문서 관련성 판별 결과."""
    binary_score: str = Field(description="관련 있으면 'yes', 없으면 'no'")

# 용도별 LLM
llm_generate = ChatOpenAI(model="gpt-5-mini", temperature=0.1)
llm_grade    = ChatOpenAI(model="gpt-5-mini", temperature=0)
llm_rewrite  = ChatOpenAI(model="gpt-5-mini", temperature=0.3)

# Generate Chain
generate_prompt = ChatPromptTemplate.from_messages([
    ("system", GENERATE_SYSTEM), ("human", GENERATE_HUMAN),
])
generate_chain = generate_prompt | llm_generate | StrOutputParser()

# Grade Chain (Structured Output)
grade_prompt = ChatPromptTemplate.from_messages([
    ("system", GRADE_SYSTEM), ("human", GRADE_HUMAN),
])
grade_chain = grade_prompt | llm_grade.with_structured_output(GradeAnswer)

# Rewrite Chain
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", REWRITE_SYSTEM), ("human", REWRITE_HUMAN),
])
rewrite_chain = rewrite_prompt | llm_rewrite | StrOutputParser()


# --- 배치 평가 체인 ---
class BatchGradeAnswer(BaseModel):
    """여러 문서의 관련성을 한꺼번에 판별한 결과."""
    grades: List[str] = Field(
        description="각 문서에 대한 관련성 판별 결과 리스트 ('yes' 또는 'no')"
    )

batch_grade_prompt = ChatPromptTemplate.from_messages([
    ("system", BATCH_GRADE_SYSTEM), ("human", BATCH_GRADE_HUMAN),
])
batch_grade_chain = batch_grade_prompt | llm_grade.with_structured_output(BatchGradeAnswer)

print("LCEL 체인 구성 완료")
print(f"  generate_chain  입력: {generate_prompt.input_variables}")
print(f"  grade_chain     입력: {grade_prompt.input_variables}")
print(f"  rewrite_chain   입력: {rewrite_prompt.input_variables}")
# --- Faithfulness / Answer Relevancy 평가 체인 ---
class FaithfulnessScore(BaseModel):
    """답변 충실도 평가 결과."""
    score: float = Field(description="충실도 점수 (0.0 ~ 1.0)")
    reasoning: str = Field(description="평가 근거")

class RelevancyScore(BaseModel):
    """답변 적합성 평가 결과."""
    score: float = Field(description="적합성 점수 (0.0 ~ 1.0)")
    reasoning: str = Field(description="평가 근거")

llm_eval = ChatOpenAI(model="gpt-5-mini", temperature=0)

faithfulness_prompt = ChatPromptTemplate.from_messages([
    ("system", FAITHFULNESS_SYSTEM), ("human", FAITHFULNESS_HUMAN),
])
faithfulness_chain = faithfulness_prompt | llm_eval.with_structured_output(FaithfulnessScore)

relevancy_prompt = ChatPromptTemplate.from_messages([
    ("system", RELEVANCY_SYSTEM), ("human", RELEVANCY_HUMAN),
])
relevancy_chain = relevancy_prompt | llm_eval.with_structured_output(RelevancyScore)

print("평가 체인 추가 완료: faithfulness_chain, relevancy_chain")

LCEL 체인 구성 완료
  generate_chain  입력: ['citations', 'context', 'question']
  grade_chain     입력: ['context', 'question']
  rewrite_chain   입력: ['question']


In [ ]:
# === Faithfulness & Answer Relevancy 평가 함수 ===
import asyncio


async def score_faithfulness(answer: str, contexts: list[str]) -> dict:
    """답변의 충실도(Faithfulness)를 평가한다.

    답변이 검색된 컨텍스트에만 근거하는지 LLM-as-judge로 측정.
    """
    if not answer or not contexts:
        return {"score": 0.0, "reasoning": "답변 또는 컨텍스트 없음"}

    context_str = "\n\n---\n\n".join(
        f"[문서 {i+1}]\n{ctx[:800]}" for i, ctx in enumerate(contexts[:5])
    )

    try:
        result = await faithfulness_chain.ainvoke({
            "context": context_str,
            "answer": answer,
        })
        return {"score": round(result.score, 2), "reasoning": result.reasoning}
    except Exception as e:
        return {"score": 0.0, "reasoning": f"평가 실패: {e}"}


async def score_answer_relevancy(question: str, answer: str) -> dict:
    """답변의 질문 적합성(Answer Relevancy)을 평가한다.

    답변이 질문의 핵심 의도에 정확히 응답하는지 LLM-as-judge로 측정.
    """
    if not question or not answer:
        return {"score": 0.0, "reasoning": "질문 또는 답변 없음"}

    try:
        result = await relevancy_chain.ainvoke({
            "question": question,
            "answer": answer,
        })
        return {"score": round(result.score, 2), "reasoning": result.reasoning}
    except Exception as e:
        return {"score": 0.0, "reasoning": f"평가 실패: {e}"}


async def evaluate_rag_response(
    question: str,
    answer: str,
    contexts: list[str],
    citations: list[str],
) -> dict:
    """RAG 응답의 종합 품질을 평가한다 (Faithfulness + Relevancy 병렬)."""
    faith_result, relev_result = await asyncio.gather(
        score_faithfulness(answer, contexts),
        score_answer_relevancy(question, answer),
    )
    return {
        "faithfulness": faith_result,
        "relevancy": relev_result,
        "avg_score": round(
            (faith_result["score"] + relev_result["score"]) / 2, 2
        ),
    }


def save_evaluation_report(results: list[dict], output_path: str = None) -> str:
    """평가 결과를 Excel 파일로 저장한다."""
    if output_path is None:
        output_path = str(PROJECT_ROOT / "evaluation_report.xlsx")

    rows = []
    for r in results:
        faith = r.get("faithfulness", {})
        relev = r.get("relevancy", {})
        rows.append({
            "번호": r.get("번호", ""),
            "질문": r.get("질문", ""),
            "답변(요약)": r.get("답변", "")[:200],
            "출처": ", ".join(r.get("출처", [])[:3]),
            "검색관련성": r.get("관련성", 0),
            "Faithfulness": faith.get("score", 0),
            "Faith근거": faith.get("reasoning", ""),
            "Relevancy": relev.get("score", 0),
            "Relev근거": relev.get("reasoning", ""),
            "평균점수": r.get("avg_score", 0),
            "재시도": r.get("재시도", 0),
            "소요시간(s)": r.get("소요시간(s)", 0),
        })

    df = pd.DataFrame(rows)
    df.to_excel(output_path, index=False, engine="openpyxl")
    print(f"평가 리포트 저장: {output_path}")
    return output_path


print("평가 함수 정의 완료: score_faithfulness, score_answer_relevancy, evaluate_rag_response")
print("  save_evaluation_report -> evaluation_report.xlsx")

---
## 5. LangGraph — Self-Corrective RAG

검색 → 관련성 평가 → 조건부 답변 생성/재검색 흐름을 구성한다.
```
retrieve → grade_documents → (조건부)
                                ├─ 관련성 충분 → generate → END
                                └─ 관련성 부족 → transform_query → retrieve (재검색)
```

In [21]:
# === GraphState 정의 ===

class GraphState(TypedDict):
    """RAG 파이프라인의 공유 상태."""
    question: str              # 사용자 질문
    chat_history: List[str]    # 대화 기록
    filters: dict              # 검색 필터
    retrieved_contexts: List[str]  # 검색된 문서 원문
    final_answer: str          # 최종 답변
    citations_used: List[str]  # 출처 (파일명)
    errors: List[str]          # 에러 메시지
    relevance_score: float     # 관련성 점수
    retry_count: int           # 재시도 횟수
    latency: float             # 검색 소요 시간
    hit_position: int          # 정답 문서 순위 (평가용)

print("GraphState 정의 완료")
print(f"  필드: {list(GraphState.__annotations__.keys())}")

GraphState 정의 완료
  필드: ['question', 'chat_history', 'filters', 'retrieved_contexts', 'final_answer', 'citations_used', 'errors', 'relevance_score', 'retry_count', 'latency', 'hit_position']


In [22]:
# === LangGraph 노드 함수 (비동기) ===
import asyncio

MAX_RETRIES = 1
STREAMING = True  # True: 답변이 실시간 토큰 단위로 출력됨


async def retrieve(state: GraphState) -> dict:
    """비동기로 벡터스토어에서 관련 문서를 검색한다."""
    question = state.get("question", "")
    filters = state.get("filters", {})
    result = await retrieve_for_graph(question, filters)
    return result


async def generate(state: GraphState) -> dict:
    """비동기로 검색된 컨텍스트 기반 답변을 생성한다. STREAMING=True면 실시간 출력."""
    question = state.get("question", "")
    contexts = state.get("retrieved_contexts", [])
    citations = state.get("citations_used", [])

    if not contexts:
        return {
            "final_answer": "관련 문서를 찾을 수 없습니다.",
            "errors": [*state.get("errors", []), "컨텍스트 없음"],
        }

    truncated = _truncate_contexts(contexts)
    context_str = "\n\n---\n\n".join(
        f"[문서 {i+1}]\n{ctx}" for i, ctx in enumerate(truncated)
    )
    citations_str = ", ".join(dict.fromkeys(citations))

    prompt_input = {
        "context": context_str,
        "citations": citations_str,
        "question": question,
    }

    try:
        if STREAMING:
            prompt_value = await generate_prompt.ainvoke(prompt_input)
            chunks = []
            async for chunk in llm_generate.astream(prompt_value):
                token = chunk.content
                print(token, end="", flush=True)
                chunks.append(token)
            print()
            answer = "".join(chunks)
        else:
            answer = await generate_chain.ainvoke(prompt_input)
    except Exception as e:
        return {
            "final_answer": "답변 생성 중 오류가 발생했습니다.",
            "errors": [*state.get("errors", []), f"LLM 호출 실패: {e}"],
        }
    return {"final_answer": answer}


async def _async_grade_one(question: str, doc: str, idx: int) -> str:
    """개별 문서를 비동기로 평가한다."""
    try:
        result = await grade_chain.ainvoke({"question": question, "context": doc})
        grade = result.binary_score
    except Exception:
        grade = "no"
    print(f"  문서 {idx}: {grade}")
    return grade


async def _fallback_grade_individually(question: str, documents: list) -> dict:
    """개별 문서 평가 폴백 — 모든 문서를 비동기 병렬로 평가한다."""
    tasks = [
        _async_grade_one(question, doc, i + 1)
        for i, doc in enumerate(documents)
    ]
    grades = await asyncio.gather(*tasks)
    relevant_count = sum(1 for g in grades if g.strip().lower() == "yes")
    score = relevant_count / len(documents)
    print(f"--- 병렬 개별 검수 완료: {relevant_count}/{len(documents)} 관련, 점수 {score:.2f} ---")
    return {"relevance_score": score}


async def grade_documents(state: GraphState) -> dict:
    """비동기로 검색된 문서의 관련성을 배치 LLM 호출로 평가한다."""
    question = state.get("question", "")
    documents = state.get("retrieved_contexts", [])

    if not documents:
        print("--- 검수: 문서 없음 ---")
        return {"relevance_score": 0.0}

    numbered_docs = "\n".join(
        f"[문서 {i+1}]\n{doc[:500]}" for i, doc in enumerate(documents)
    )

    try:
        print("--- 배치 평가 시작 ---")
        batch_result = await batch_grade_chain.ainvoke({
            "question": question,
            "numbered_docs": numbered_docs,
        })
        grades = batch_result.grades

        if len(grades) != len(documents):
            print(f"  배치 결과 수 불일치: {len(grades)} != {len(documents)}, 폴백 수행")
            return await _fallback_grade_individually(question, documents)

        relevant_count = sum(1 for g in grades if g.strip().lower() == "yes")
        for i, g in enumerate(grades):
            print(f"  문서 {i+1}: {g.strip().lower()}")

        score = relevant_count / len(documents)
        print(f"--- 배치 검수 완료: {relevant_count}/{len(documents)} 관련, 점수 {score:.2f} ---")
        return {"relevance_score": score}

    except Exception as e:
        print(f"  배치 평가 실패 ({e}), 병렬 개별 평가로 폴백")
        return await _fallback_grade_individually(question, documents)


async def transform_query(state: GraphState) -> dict:
    """비동기로 질문을 재작성하여 검색 품질을 높인다."""
    original = state.get("question", "")
    retry_count = state.get("retry_count", 0) + 1

    try:
        rewritten = await rewrite_chain.ainvoke({"question": original})
    except Exception:
        rewritten = original

    print(f"--- 질문 재작성 (retry {retry_count}) ---")
    print(f"  원래: {original}")
    print(f"  변환: {rewritten}")
    return {"question": rewritten, "retry_count": retry_count}


def decide_to_generate(state: GraphState) -> str:
    """관련성에 따라 다음 노드를 결정한다."""
    if state.get("retry_count", 0) >= MAX_RETRIES:
        print(f"--- 최대 재시도({MAX_RETRIES}회) 도달 → 답변 생성 ---")
        return "generate"
    if state.get("relevance_score", 0) >= 0.5:
        print("--- 관련성 충분 → 답변 생성 ---")
        return "generate"
    print("--- 관련성 부족 → 질문 재작성 ---")
    return "transform_query"


print("비동기 노드 함수 정의 완료 (Streaming 모드:", "ON" if STREAMING else "OFF", ")")
print(f"  MAX_RETRIES={MAX_RETRIES}")

비동기 노드 함수 정의 완료 (Streaming 모드: ON )
  MAX_RETRIES=1


In [23]:
# === MVP 단순 RAG 그래프 ===

def build_simple_graph():
    """retrieve → generate → END"""
    graph = StateGraph(GraphState)
    graph.add_node("retrieve", retrieve)
    graph.add_node("generate", generate)
    graph.add_edge(START, "retrieve")
    graph.add_edge("retrieve", "generate")
    graph.add_edge("generate", END)
    return graph.compile()

simple_graph = build_simple_graph()
print("Simple RAG 그래프 빌드 완료")

Simple RAG 그래프 빌드 완료


In [24]:
# === Self-Corrective RAG 그래프 (최적화) ===

# 검색 점수가 높으면 grade 생략 → 바로 generate (LLM 1회 절약)
SKIP_GRADE_THRESHOLD = 0.012  # hybrid RRF 상위3 평균 기준


def decide_grade_or_skip(state: GraphState) -> str:
    """검색 점수가 높으면 grade를 건너뛰고 바로 답변 생성."""
    score = state.get("relevance_score", 0)
    if score >= SKIP_GRADE_THRESHOLD:
        print(f"--- 검색 점수 높음({score:.4f} >= {SKIP_GRADE_THRESHOLD}) → grade 생략, 바로 생성 ---")
        return "generate"
    print(f"--- 검색 점수 낮음({score:.4f}) → grade 평가 진행 ---")
    return "grade_documents"


def build_self_corrective_graph():
    """retrieve → (고점수: generate | 저점수: grade → generate/rewrite)"""
    graph = StateGraph(GraphState)
    graph.add_node("retrieve", retrieve)
    graph.add_node("grade_documents", grade_documents)
    graph.add_node("generate", generate)
    graph.add_node("transform_query", transform_query)

    graph.add_edge(START, "retrieve")
    graph.add_conditional_edges(
        "retrieve",
        decide_grade_or_skip,
        {"generate": "generate", "grade_documents": "grade_documents"},
    )
    graph.add_conditional_edges(
        "grade_documents",
        decide_to_generate,
        {"generate": "generate", "transform_query": "transform_query"},
    )
    graph.add_edge("transform_query", "retrieve")
    graph.add_edge("generate", END)
    return graph.compile()

corrective_graph = build_self_corrective_graph()
print("Self-Corrective RAG 그래프 빌드 완료 (grade skip 최적화)")
print(f"  SKIP_GRADE_THRESHOLD={SKIP_GRADE_THRESHOLD}")
print(f"  MAX_RETRIES={MAX_RETRIES}")

Self-Corrective RAG 그래프 빌드 완료 (grade skip 최적화)
  SKIP_GRADE_THRESHOLD=0.012
  MAX_RETRIES=1


---
## 6. 파이프라인 실행

### 6-1. Simple RAG 실행
단순 retrieve → generate 흐름으로 질문에 답변한다.

In [25]:
# Simple RAG 실행 (비동기)
init_state = {
    "question": "이 사업의 총 예산은 얼마인가요?",
    "chat_history": [],
    "filters": {},
    "retrieved_contexts": [],
    "final_answer": "",
    "citations_used": [],
    "errors": [],
    "relevance_score": 0.0,
    "retry_count": 0,
    "latency": 0.0,
    "hit_position": 0,
}

print("=== Simple RAG 실행 ===")
result = await simple_graph.ainvoke(init_state)

print(f"\n질문: {result['question']}")
print(f"\n답변:\n{result['final_answer']}")
print(f"\n출처: {list(dict.fromkeys(result['citations_used']))}")
if result.get("errors"):
    print(f"에러: {result['errors']}")

=== Simple RAG 실행 ===
핵심 요약
- 총사업비는 977,240,000원입니다.

세부 사항
- 문서 표기: "고정단가 확인 필수 (단위 : 원)"에 따른 총사업비: 977,240,000
- 항목별 내역(문서에 표기된 금액, 단위: 원)
  - 1. 직접비: 604,600,760
  - 2. 경비: 195,765,000
  - 3. 초청워크숍: 118,239,840
  - 4. 성과관리: 58,634,400
- 환율 참고: 1USD = 1,300원 (문서 표기)

[출처]
- 한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아).hwp

질문: 이 사업의 총 예산은 얼마인가요?

답변:
핵심 요약
- 총사업비는 977,240,000원입니다.

세부 사항
- 문서 표기: "고정단가 확인 필수 (단위 : 원)"에 따른 총사업비: 977,240,000
- 항목별 내역(문서에 표기된 금액, 단위: 원)
  - 1. 직접비: 604,600,760
  - 2. 경비: 195,765,000
  - 3. 초청워크숍: 118,239,840
  - 4. 성과관리: 58,634,400
- 환율 참고: 1USD = 1,300원 (문서 표기)

[출처]
- 한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아).hwp

출처: ['서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 구축.hwp', '한국재정정보원_e나라도움 업무시스템 웹 접근성 컨설팅.hwp', '한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp', '한국연구재단_2024년 기초학문자료센터 시스템 운영 및 연구성과물 DB구.hwp', '한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp']


### 6-2. Self-Corrective RAG 실행
관련성 평가 → 재검색 루프가 포함된 고급 흐름.

In [26]:
# Self-Corrective RAG 실행 (비동기)
corrective_state = {
    "question": "입찰 참가 자격 요건은 무엇인가요?",
    "chat_history": [],
    "filters": {},
    "retrieved_contexts": [],
    "final_answer": "",
    "citations_used": [],
    "errors": [],
    "relevance_score": 0.0,
    "retry_count": 0,
    "latency": 0.0,
    "hit_position": 0,
}

print("=== Self-Corrective RAG 실행 ===")
result_sc = await corrective_graph.ainvoke(corrective_state)

print(f"\n질문: {result_sc['question']}")
print(f"관련성 점수: {result_sc['relevance_score']}")
print(f"재시도 횟수: {result_sc['retry_count']}")
print(f"\n답변:\n{result_sc['final_answer']}")
print(f"\n출처: {list(dict.fromkeys(result_sc['citations_used']))}")

=== Self-Corrective RAG 실행 ===
--- 검색 점수 낮음(0.0115) → grade 평가 진행 ---
--- 배치 평가 시작 ---
  문서 1: yes
  문서 2: yes
  문서 3: no
  문서 4: yes
  문서 5: no
--- 배치 검수 완료: 3/5 관련, 점수 0.60 ---
--- 관련성 충분 → 답변 생성 ---
핵심 요약
- 제공된 RFP 문서들에서 공통적으로 요구하는 입찰 참가자격은 “지방자치단체를 당사자로 하는 계약에 관한 법률 시행령 제13조(입찰의 참가자격) 및 동법 시행규칙 제14조(입찰 참가자격 요건의 증명)”에 따른 경쟁입찰 참가자격 보유를 기본으로 하고, 사업별로 소프트웨어사업자 등록, 직접생산 확인(제품·품목 코드), 중소기업·소상공인 확인, 조달청·나라장터 등록·증명서류 보유 등 추가 요건이 명시되어 있습니다. 상세 요건은 아래에 문서별·항목별로 정리했습니다.

세부 사항
- 기본 법적 자격
  - 「지방자치단체를 당사자로 하는 계약에 관한 법률 시행령」 제13조 및 동법 시행규칙 제14조에 의한 경쟁입찰 참가자격을 갖출 것. (문서 1, 2, 4 등)

- 소프트웨어사업자 등록 관련
  - 「소프트웨어 진흥법」 제24조에 의거 소프트웨어사업자(분야: 컴퓨터 관련 서비스사업, 업종코드: 1468)로 등록되어야 함. (문서 1, 2, 4)
  - 일부 사업은 소프트웨어 진흥법 시행령·시행규칙(예: 제53조·제17조) 기준으로 등록 시점(입찰서 제출일/접수마감일)까지 등록되어 있어야 함. (문서 4)

- 직접생산 확인(중소기업 관련)
  - 「중소기업제품 구매촉진 및 판로지원에 관한 법률」 및 동법 시행령에 따른 직접생산 확인증명서 보유 필요.
    - 예시 표기: 제품명: 소프트웨어엔지니어링업, 세부품명 : 정보시스템개발서비스(8111159901) — 유효기간 내에 있어야 함. (문서 1)
    - 일부 공고는 정보시스템개발서비스 및 정보시스템유지관리서비스 등으로 명시(예: 정보시스템개발서비

---
## 7. 검색 품질 평가

다양한 질문으로 Hit Rate, 응답 시간을 측정한다.

| 지표 | 목표 |
| :--- | :--- |
| Hit Rate @k | >= 0.8 |
| 평균 Latency | < 2초 |

In [27]:
eval_queries = [
    "이 사업의 총 예산은?",
    "제안서 제출 기한이 언제인가요?",
    "입찰 참가 자격 요건은?",
    "사업 범위와 주요 요구사항은?",
    "발주 기관이 어디인가요?",
    "협상에 의한 계약 조건은?",
    "기술 요구사항 중 보안 관련 조건은?",
    "하도급 제한 조건이 있나요?",
]

latencies = []
result_counts = []

print("=== 배치 검색 평가 (비동기) ===")
for q in eval_queries:
    r = await retrieve_for_graph(q)
    n = len(r["retrieved_contexts"])
    lat = r["latency"]
    latencies.append(lat)
    result_counts.append(n)
    status = "OK" if n > 0 else "MISS"
    print(f"  [{status}] {q:30s} -> {n}개, {lat}s")

hit_rate = sum(1 for c in result_counts if c > 0) / len(eval_queries)
avg_latency = sum(latencies) / len(latencies)

print(f"\n=== 평가 결과 ===")
print(f"Hit Rate @{DEFAULT_K}: {hit_rate:.2f} (목표 >= 0.8)")
print(f"평균 Latency: {avg_latency:.4f}s (목표 < 2s)")

=== 배치 검색 평가 (비동기) ===
  [OK] 이 사업의 총 예산은?                   -> 5개, 0.2434s
  [OK] 제안서 제출 기한이 언제인가요?              -> 5개, 0.1188s
  [OK] 입찰 참가 자격 요건은?                  -> 5개, 0.1331s
  [OK] 사업 범위와 주요 요구사항은?               -> 5개, 0.1812s
  [OK] 발주 기관이 어디인가요?                  -> 5개, 0.1394s
  [OK] 협상에 의한 계약 조건은?                 -> 5개, 0.1401s
  [OK] 기술 요구사항 중 보안 관련 조건은?           -> 5개, 0.18s
  [OK] 하도급 제한 조건이 있나요?                -> 5개, 0.1207s

=== 평가 결과 ===
Hit Rate @5: 1.00 (목표 >= 0.8)
평균 Latency: 0.1571s (목표 < 2s)


In [28]:
# Self-Corrective RAG E2E 평가 (비동기)
print("=== Self-Corrective RAG E2E 평가 ===\n")

eval_qa = [
    "이 사업의 총 예산은 얼마인가요?",
    "제안서 제출 마감일은 언제인가요?",
    "입찰 참가 자격 요건은 무엇인가요?",
]

for q in eval_qa:
    state = {
        "question": q,
        "chat_history": [],
        "filters": {},
        "retrieved_contexts": [],
        "final_answer": "",
        "citations_used": [],
        "errors": [],
        "relevance_score": 0.0,
        "retry_count": 0,
        "latency": 0.0,
        "hit_position": 0,
    }
    start = time.time()
    result = await corrective_graph.ainvoke(state)
    elapsed = time.time() - start

    print(f"Q: {q}")
    print(f"A: {result['final_answer'][:200]}...")
    print(f"출처: {list(dict.fromkeys(result['citations_used']))[:3]}")
    print(f"관련성: {result['relevance_score']:.2f}, 재시도: {result['retry_count']}, 소요: {elapsed:.2f}s")
    print("-" * 60)

=== Self-Corrective RAG E2E 평가 ===

--- 검색 점수 낮음(0.0115) → grade 평가 진행 ---
--- 배치 평가 시작 ---
  문서 1: no
  문서 2: no
  문서 3: yes
  문서 4: no
  문서 5: no
--- 배치 검수 완료: 1/5 관련, 점수 0.20 ---
--- 관련성 부족 → 질문 재작성 ---
--- 질문 재작성 (retry 1) ---
  원래: 이 사업의 총 예산은 얼마인가요?
  변환: 사업 총 예산(추정가격)?
--- 검색 점수 낮음(0.0115) → grade 평가 진행 ---
--- 배치 평가 시작 ---
  문서 1: yes
  문서 2: yes
  문서 3: no
  문서 4: no
  문서 5: no
--- 배치 검수 완료: 2/5 관련, 점수 0.40 ---
--- 최대 재시도(1회) 도달 → 답변 생성 ---
요약
- 제공된 문서들 중 사업 총예산(추정가격)이 명시된 것은 광주과학기술원 학사시스템 기능개선 사업 문서뿐이며, 그 금액은 "157,300,000원(VAT 포함)"입니다.
- 나머지 문서들에서는 사업 총예산(추정가격)을 확인할 수 없습니다.

세부사항
- 광주과학기술원 학사시스템 기능개선 사업(문서 3)
  - 사업비 : 157,300,000원(VAT 포함)
- 서민금융진흥원 서민금융 채팅 상담시스템 구축(문서 1)
  - 대상 업체별 사업금액의 하한만 제시되어 있으나, 사업 총예산(추정가격)은 제공된 문서에서 확인할 수 없습니다.
  - 제시된 하한: 매출액 8천억 원 이상인 대기업 — 80억 원 이상; 매출액 8천억 원 미만인 대기업 — 40억 원 이상
- (긴급) 모잠비크 마푸토 ITS 구축사업 F/S 용역(문서 2)
  - 총 추정사업비 산정 표가 있으나 수치는 "ooo"로 표기되어 있어 구체적 금액은 제공된 문서에서 확인할 수 없습니다.
- 한국발명진흥회 2024년 시스템 개편 용역(문서 4)
  - 재무·실적 서식 등이 포함되어 있으나 사업 총예산(

---
### 7-1. 수동 검증 프레임워크

LLM 비용 없이 **로그 기반**으로 사람이 직접 RAG 품질을 검증하는 프레임워크.

- 각 질문에 대한 답변, 검색 문서, 출처, 관련성 점수를 상세 출력
- 수동 체크리스트로 Faithfulness, Relevancy, Citation, Hallucination 판별


In [29]:
# === 종합 평가: 검색 + Faithfulness + Relevancy (비동기) ===
_prev_streaming = STREAMING
STREAMING = False

manual_eval_queries = [
    "이 사업의 총 예산은 얼마인가요?",
    "제안서 제출 마감일은 언제인가요?",
    "입찰 참가 자격 요건은 무엇인가요?",
    "사업 범위와 주요 요구사항은 무엇인가요?",
    "발주 기관이 어디인가요?",
    "협상에 의한 계약 조건은 무엇인가요?",
    "기술 요구사항 중 보안 관련 조건은?",
    "하도급 제한 조건이 있나요?",
]

manual_eval_results = []

print("=== 종합 평가 (검색 + Faithfulness + Relevancy) 시작 ===\n")

for idx, q in enumerate(manual_eval_queries, 1):
    init_state = {
        "question": q,
        "chat_history": [],
        "filters": {},
        "retrieved_contexts": [],
        "final_answer": "",
        "citations_used": [],
        "errors": [],
        "relevance_score": 0.0,
        "retry_count": 0,
        "latency": 0.0,
        "hit_position": 0,
    }

    start = time.time()
    result = await corrective_graph.ainvoke(init_state)
    elapsed = round(time.time() - start, 2)

    # Faithfulness + Relevancy 병렬 평가
    eval_scores = await evaluate_rag_response(
        question=q,
        answer=result["final_answer"],
        contexts=result["retrieved_contexts"],
        citations=result["citations_used"],
    )

    record = {
        "번호": idx,
        "질문": q,
        "답변": result["final_answer"],
        "출처": list(dict.fromkeys(result["citations_used"])),
        "관련성": result["relevance_score"],
        "faithfulness": eval_scores["faithfulness"],
        "relevancy": eval_scores["relevancy"],
        "avg_score": eval_scores["avg_score"],
        "재시도": result["retry_count"],
        "소요시간(s)": elapsed,
        "검색문서": result["retrieved_contexts"],
        "에러": result["errors"],
    }
    manual_eval_results.append(record)

    faith_s = eval_scores["faithfulness"]["score"]
    relev_s = eval_scores["relevancy"]["score"]
    print(f"[{idx}/{len(manual_eval_queries)}] Q: {q}")
    print(f"  답변: {result['final_answer'][:150]}...")
    print(f"  출처: {record['출처'][:3]}")
    print(f"  검색관련성: {record['관련성']:.2f} | Faithfulness: {faith_s:.2f} | Relevancy: {relev_s:.2f}")
    print(f"  재시도: {record['재시도']} | 소요: {elapsed}s")
    print("-" * 70)

# pandas DataFrame 요약 테이블
df_eval = pd.DataFrame([
    {
        "번호": r["번호"],
        "질문": r["질문"][:30],
        "검색관련성": r["관련성"],
        "Faithfulness": r["faithfulness"]["score"],
        "Relevancy": r["relevancy"]["score"],
        "평균점수": r["avg_score"],
        "재시도": r["재시도"],
        "소요시간(s)": r["소요시간(s)"],
    }
    for r in manual_eval_results
])

print("\n=== 종합 평가 요약 테이블 ===")
from IPython.display import display as ipy_display
ipy_display(df_eval)

print(f"\n평균 Faithfulness: {df_eval['Faithfulness'].mean():.2f}")
print(f"평균 Relevancy: {df_eval['Relevancy'].mean():.2f}")
print(f"평균 종합 점수: {df_eval['평균점수'].mean():.2f}")
print(f"평균 소요시간: {df_eval['소요시간(s)'].mean():.2f}s")

# Excel 리포트 자동 저장
report_path = save_evaluation_report(manual_eval_results)

STREAMING = _prev_streaming

=== 수동 검증 데이터 수집 시작 ===

--- 검색 점수 낮음(0.0115) → grade 평가 진행 ---
--- 배치 평가 시작 ---
  문서 1: no
  문서 2: no
  문서 3: yes
  문서 4: no
  문서 5: no
--- 배치 검수 완료: 1/5 관련, 점수 0.20 ---
--- 관련성 부족 → 질문 재작성 ---
--- 질문 재작성 (retry 1) ---
  원래: 이 사업의 총 예산은 얼마인가요?
  변환: 사업 총예산(추정가격·사업비)?
--- 검색 점수 높음(0.0157 >= 0.012) → grade 생략, 바로 생성 ---
[1/8] Q: 이 사업의 총 예산은 얼마인가요?
  답변: 핵심 요약
- 제공된 RFP 문서들에는 사업의 총예산(추정가격·사업비) 금액이 명시되어 있지 않습니다. 따라서 총예산 수치는 제공된 문서에서 확인할 수 없습니다.

세부 사항
- 문서 1: 사업비 항목별 구성(직접공사비, 컨설팅비, 간접사업비 등) 및 산식 형태의 표가...
  출처: ['한국수출입은행_(긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업.hwp', '사단법인 보험개발원_실손보험 청구 전산화 시스템 구축 사업.hwp', '인천공항운영서비스(주)_인천공항운영서비스㈜ 차세대 ERP시스템 구축 .hwp']
  관련성: 0.02 | 재시도: 1 | 소요: 32.74s
  [문서 1] [사업명: (긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업 사업타당성조사(F/S) 용역] [발주기관: 한국수출입은행]
EDCF 및 사업국 기준에 따른 물량예비비 및 가격예비비 산정
(간접사업비) 예비비, 사업관리비, 토지보상비, 세금 등 산정
물량예비비 및 가격예비비 산정
수원국 세금 관련체계 (관세, VAT, Withholding Tax 등)
...
  [문서 2] [사업명: (긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업 사업타당성조사(F/S) 용역] [발주기관: 한국수출입은행]
[Part 7] 사업비 산정
□ 직접사업

,번호,질문,관련성,재시도,소요시간(s),출처수,답변길이
0,1,이 사업의 총 예산은 얼마인가요?,0.0157,1,32.74,4,715
1,2,제안서 제출 마감일은 언제인가요?,0.0000,1,47.09,4,889
2,3,입찰 참가 자격 요건은 무엇인가요?,0.6000,0,45.61,5,2511
3,4,사업 범위와 주요 요구사항은 무엇인가요?,0.6000,0,44.25,2,1743
4,5,발주 기관이 어디인가요?,1.0000,0,24.66,4,429
5,6,협상에 의한 계약 조건은 무엇인가요?,0.0140,0,49.01,5,2184
6,7,기술 요구사항 중 보안 관련 조건은?,1.0000,0,41.86,5,2878
7,8,하도급 제한 조건이 있나요?,0.0143,0,29.13,4,1488



평균 소요시간: 39.29s
평균 관련성: 0.41


In [30]:
# === 수동 검증 체크리스트 ===
# 각 질문별로 사람이 직접 판단할 수 있는 항목 출력

print("=== 수동 품질 검증 체크리스트 ===")

SEP = "=" * 70

for r in manual_eval_results:
    print()
    print(SEP)
    print(f"[질문 {r['번호']}] {r['질문']}")
    print(SEP)

    # 답변 전문
    print()
    print("[답변 전문]")
    print(r["답변"])

    # 검색 문서 원문
    print()
    print("[검색 문서 원문 (상위 5개)]")
    for di, doc in enumerate(r["검색문서"][:5], 1):
        print(f"  --- 문서 {di} ---")
        print(f"  {doc[:300]}")
        print()

    # 출처 정보
    print(f"[출처] {r['출처']}")
    print(f"[관련성] {r['관련성']:.2f} | [재시도] {r['재시도']}회")

    # 체크리스트
    print()
    print("--- 검증 체크리스트 ---")
    print("  [ ] [Faithfulness]  답변이 검색 문서 내용에만 근거하는가?")
    print("  [ ] [Relevancy]     질문에 대한 적절한 답변인가?")
    print("  [ ] [Citation]      출처가 정확한가? (답변 내용이 해당 문서에 존재)")
    print("  [ ] [Hallucination] 문서에 없는 내용을 지어냈는가? (No면 통과)")
    print()


=== 수동 품질 검증 체크리스트 ===

[질문 1] 이 사업의 총 예산은 얼마인가요?

[답변 전문]
핵심 요약
- 제공된 RFP 문서들에는 사업의 총예산(추정가격·사업비) 금액이 명시되어 있지 않습니다. 따라서 총예산 수치는 제공된 문서에서 확인할 수 없습니다.

세부 사항
- 문서 1: 사업비 항목별 구성(직접공사비, 컨설팅비, 간접사업비 등) 및 산식 형태의 표가 제시되어 있으나, 합계 금액(총 추정사업비)은 기재되어 있지 않습니다.
- 문서 2: "총 추정사업비 산정"이 과업범위에 포함되어 있고, 직접사업비·간접사업비 산정 항목 및 요구 산출물이 상세히 기술되어 있으나 실제 추정금액은 제공되어 있지 않습니다.
- 문서 3~5: 가격평가 방법, 평가기준, 실적·재무·인력 제출 양식 등 평가 및 제출요건 관련 내용이 포함되어 있으나, 본 사업의 총예산(추정가격·사업비) 수치는 포함되어 있지 않습니다.
- 결론: 문서 전체를 통합해도 총예산(추정가격·사업비) 금액은 제공되어 있지 않으므로 산출된 숫자를 제시할 수 없습니다.

제공된 문서에서 해당 정보를 확인할 수 없습니다.

[출처]
- 한국수출입은행_ (긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업 사업타당성조사(F/S) 용역 (문서 1, 문서 2)
- 사단법인 보험개발원_ 실손보험 청구 전산화 시스템 구축 사업 (문서 3)
- 인천공항운영서비스(주)_ 인천공항운영서비스㈜ 차세대 ERP시스템 구축 사업 (문서 4)
- 울산광역시_ 2024년 버스정보시스템 확대 구축 및 기능개선 용역 (문서 5)

[검색 문서 원문 (상위 5개)]
  --- 문서 1 ---
  [사업명: (긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업 사업타당성조사(F/S) 용역] [발주기관: 한국수출입은행]
EDCF 및 사업국 기준에 따른 물량예비비 및 가격예비비 산정
(간접사업비) 예비비, 사업관리비, 토지보상비, 세금 등 산정
물량예비비 및 가격예비비 산정
수원국 세금 관련체계 (관세, VAT, Withhol

In [31]:
# === 배치 평가 성능 비교 ===
# 배치 평가 적용 후 응답 시간 측정 및 이전 기록과 비교

# 이전 기록 (개별 평가 시 측정값) — 직접 기록하거나 이전 실행 결과 입력
PREVIOUS_AVG_LATENCY = None  # 이전 평균 소요시간 (초), 없으면 None

if manual_eval_results:
    current_latencies = [r['소요시간(s)'] for r in manual_eval_results]
    current_avg = sum(current_latencies) / len(current_latencies)
    current_min = min(current_latencies)
    current_max = max(current_latencies)

    print("=== 배치 평가 성능 측정 결과 ===")
    print(f"  평균 소요시간: {current_avg:.2f}s")
    print(f"  최소 소요시간: {current_min:.2f}s")
    print(f"  최대 소요시간: {current_max:.2f}s")
    print(f"  총 질문 수: {len(current_latencies)}")

    if PREVIOUS_AVG_LATENCY is not None:
        improvement = PREVIOUS_AVG_LATENCY - current_avg
        pct = (improvement / PREVIOUS_AVG_LATENCY) * 100
        print(f"\n=== 이전 대비 성능 비교 ===")
        print(f"  이전 평균: {PREVIOUS_AVG_LATENCY:.2f}s")
        print(f"  현재 평균: {current_avg:.2f}s")
        print(f"  개선량: {improvement:+.2f}s ({pct:+.1f}%)")
    else:
        print("\n[참고] PREVIOUS_AVG_LATENCY를 설정하면 이전 대비 성능 비교가 가능합니다.")
        print("다음 실행 시 비교를 위해 현재 평균값을 기록해두세요:")
        print(f"  PREVIOUS_AVG_LATENCY = {current_avg:.2f}")
else:
    print("수동 검증 데이터가 없습니다. 위 셀을 먼저 실행하세요.")


=== 배치 평가 성능 측정 결과 ===
  평균 소요시간: 39.29s
  최소 소요시간: 24.66s
  최대 소요시간: 49.01s
  총 질문 수: 8

[참고] PREVIOUS_AVG_LATENCY를 설정하면 이전 대비 성능 비교가 가능합니다.
다음 실행 시 비교를 위해 현재 평균값을 기록해두세요:
  PREVIOUS_AVG_LATENCY = 39.29


---
## 8. 질의응답 (Q&A)

100개 RFP 문서 전체를 대상으로 질의응답을 수행한다.

### 사용법
```python
# 기본 질문
ask("이 사업의 총 예산은?")

# 특정 기관 필터
ask("사업 개요를 알려줘", filters={"기관": "국방부"})

# 특정 파일 대상
ask("참가 자격은?", filters={"파일": "파일명.hwp"})
```

In [32]:
# === ask() 편의 함수 (비동기) ===

async def ask(question: str, filters: dict = None, mode: str = "corrective") -> dict:
    """100개 RFP 문서를 대상으로 비동기 질의응답을 수행한다.

    Args:
        question: 사용자 질문
        filters: 검색 필터 (선택). {"기관": "...", "파일": "..."} 형태
        mode: "simple" (단순 RAG) 또는 "corrective" (Self-Corrective RAG)

    Returns:
        dict with keys: answer, sources, relevance, retries, elapsed
    """
    state = {
        "question": question,
        "chat_history": [],
        "filters": filters or {},
        "retrieved_contexts": [],
        "final_answer": "",
        "citations_used": [],
        "errors": [],
        "relevance_score": 0.0,
        "retry_count": 0,
        "latency": 0.0,
        "hit_position": 0,
    }

    graph = corrective_graph if mode == "corrective" else simple_graph
    start = time.time()
    result = await graph.ainvoke(state)
    elapsed = round(time.time() - start, 2)

    # 중복 제거된 출처 목록
    sources = list(dict.fromkeys(result.get("citations_used", [])))

    # 출력
    print(f"Q: {question}")
    if filters:
        print(f"   필터: {filters}")
    print(f"\nA: {result['final_answer']}")
    print(f"\n[출처] {', '.join(sources[:5])}")
    if mode == "corrective":
        print(f"[관련성] {result.get('relevance_score', 0):.2f}  [재시도] {result.get('retry_count', 0)}  [소요] {elapsed}s")
    else:
        print(f"[소요] {elapsed}s")

    if result.get("errors"):
        print(f"[경고] {result['errors']}")

    return {
        "answer": result["final_answer"],
        "sources": sources,
        "relevance": result.get("relevance_score", 0),
        "retries": result.get("retry_count", 0),
        "elapsed": elapsed,
        "errors": result.get("errors", []),
    }


print("ask() 비동기 함수 정의 완료")

ask() 비동기 함수 정의 완료


### 8-1. 질의응답 예시

다양한 질문 유형으로 100개 RFP 문서에 대한 Q&A를 시연한다.

In [33]:
# 예시 1: 첫 번째 데이터(한영대학교) — 사업 개요 질문
await ask("한영대학교 특성화 맞춤형 교육환경 구축 사업의 개요를 알려줘",
    filters={"파일": "한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp"})

--- 검색 점수 낮음(0.0000) → grade 평가 진행 ---
--- 배치 평가 시작 ---
  문서 1: no
  문서 2: yes
  문서 3: no
  문서 4: yes
  문서 5: yes
--- 배치 검수 완료: 3/5 관련, 점수 0.60 ---
--- 관련성 충분 → 답변 생성 ---
핵심 요약
- 사업명: 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화. 발주기관: 한영대학.
- 목적: 트랙제 기반 교육과정 도입·운영을 지원하여 다양한 진로선택 기회 제공 및 산업체 수요 맞춤형 교육과정 운영을 통한 현장실무형 인재 양성.
- 예산·기간·입찰방법 등 주요 조건: 사업예산과 사업기간, 입찰방식, 제출기한 등은 아래 세부에 정리함.

세부 사항
- 기본 정보
  - 사업명: 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 (문서 표기)
  - 발주기관: 한영대학
  - 공고번호: 20241001798 (문서 표기)
- 예산
  - 문서 표기: "사업예산 : 130,000,000원 범위 내 (VAT 포함)" (문서 2)
  - 동일 문서(요약) 표기: "사업예산: 1.3억원" (문서 4)
- 기간 및 일정
  - 사업기간: "계약일로부터 3개월 (안정화기간 1개월 포함)" (문서 2)
  - 제출기한(제안서 제출): "2024-10-15 17:00:00" (문서 4)
  - 추진단계(예시): 사업발주 및 착수 → 분석(업무 분석) → 구현(시스템 구축, 유관 시스템 연계/연동) → 시험(운영자/사용자 교육, 종합 테스트) → 운영(시스템 정상 가동) → 안정화 (문서 5)
  - 기간 및 일정은 "학교 사정과 용역대상자와의 협의에 따라 조정될 수 있음" / "발주기관 와 계약상대자의 상호협의를 통해 변경할 수 있음" (문서 2, 문서 5)
- 입찰/계약 방식 및 조건
  - 입찰방법: "제한경쟁입찰(협상에 의한 계약 체결)" (문서 2)
  - 계약 관련: 협상결과 및 관련 법령(「국가를 당사자로

{'answer': '핵심 요약\n- 사업명: 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화. 발주기관: 한영대학.\n- 목적: 트랙제 기반 교육과정 도입·운영을 지원하여 다양한 진로선택 기회 제공 및 산업체 수요 맞춤형 교육과정 운영을 통한 현장실무형 인재 양성.\n- 예산·기간·입찰방법 등 주요 조건: 사업예산과 사업기간, 입찰방식, 제출기한 등은 아래 세부에 정리함.\n\n세부 사항\n- 기본 정보\n  - 사업명: 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 (문서 표기)\n  - 발주기관: 한영대학\n  - 공고번호: 20241001798 (문서 표기)\n- 예산\n  - 문서 표기: "사업예산 : 130,000,000원 범위 내 (VAT 포함)" (문서 2)\n  - 동일 문서(요약) 표기: "사업예산: 1.3억원" (문서 4)\n- 기간 및 일정\n  - 사업기간: "계약일로부터 3개월 (안정화기간 1개월 포함)" (문서 2)\n  - 제출기한(제안서 제출): "2024-10-15 17:00:00" (문서 4)\n  - 추진단계(예시): 사업발주 및 착수 → 분석(업무 분석) → 구현(시스템 구축, 유관 시스템 연계/연동) → 시험(운영자/사용자 교육, 종합 테스트) → 운영(시스템 정상 가동) → 안정화 (문서 5)\n  - 기간 및 일정은 "학교 사정과 용역대상자와의 협의에 따라 조정될 수 있음" / "발주기관 와 계약상대자의 상호협의를 통해 변경할 수 있음" (문서 2, 문서 5)\n- 입찰/계약 방식 및 조건\n  - 입찰방법: "제한경쟁입찰(협상에 의한 계약 체결)" (문서 2)\n  - 계약 관련: 협상결과 및 관련 법령(「국가를 당사자로 하는 계약에 관한 법률」 등)에 따름 (문서 4)\n- 참가자격(문서 표기 요건 그대로)\n  - ① 「국가를 당사자로 하는 계약에 관한법률 시행령」 제12조 및 동법 시행규칙 제14조 규정에 의한 경쟁입찰 참가자격을 갖춘 업체 (문서 4)\n  

In [34]:
# 예시 2: 첫 번째 데이터 — 예산 질문
await ask("이 사업의 총 예산은 얼마인가요?",
    filters={"파일": "한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp"})

--- 검색 점수 낮음(0.0000) → grade 평가 진행 ---
--- 배치 평가 시작 ---
  문서 1: no
  문서 2: no
  문서 3: no
  문서 4: no
  문서 5: no
--- 배치 검수 완료: 0/5 관련, 점수 0.00 ---
--- 관련성 부족 → 질문 재작성 ---
--- 질문 재작성 (retry 1) ---
  원래: 이 사업의 총 예산은 얼마인가요?
  변환: 사업 총 예산(추정가격, 예산안)?
--- 검색 점수 낮음(0.0000) → grade 평가 진행 ---
--- 배치 평가 시작 ---
  문서 1: no
  문서 2: no
  문서 3: no
  문서 4: no
  문서 5: no
--- 배치 검수 완료: 0/5 관련, 점수 0.00 ---
--- 최대 재시도(1회) 도달 → 답변 생성 ---
핵심 요약
- 제공된 문서들에서는 사업 총 예산(추정가격, 예산안)을 확인할 수 없습니다.

세부 사항
- 참조 문서: 문서 1~5(제안서 제출·반환 관련 규정, 제안서 보상 여부, 검수·대금지급·계약변경·해지 조건, 누출금지 및 제재조치 등 포함).
- 문서 내용 중 확인 가능한 주요 항목 예:
  - 제안서는 반드시 직접 제출해야 하며 제출비용은 제안사 부담(문서 1).
  - 본 사업은 제안서 보상대상에 해당하지 않아 제안서 보상을 실시하지 않음(문서 1).
  - 소프트웨어 하자담보책임기간은 사업 종료일로부터 1년(문서 2).
  - 소스코드·내부망 정보 등 누출금지 대상정보 및 위반 시 입찰참가자격 제한(문서 5).
  - 사업 완료 시 감독관 입회하에 종합시험을 실시하고 규격·제안내용이 문제없이 수행되어야 함(문서 1, 4).
- 위 문서들 어느 곳에도 “사업 총 예산”, “추정가격”, “예산안” 등의 금액 정보가 명시되어 있지 않습니다. 따라서 요청하신 예산 정보는 제공된 문서에서 확인할 수 없습니다.

[출처]
- 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp (문서 1~5)
Q

{'answer': '핵심 요약\n- 제공된 문서들에서는 사업 총 예산(추정가격, 예산안)을 확인할 수 없습니다.\n\n세부 사항\n- 참조 문서: 문서 1~5(제안서 제출·반환 관련 규정, 제안서 보상 여부, 검수·대금지급·계약변경·해지 조건, 누출금지 및 제재조치 등 포함).\n- 문서 내용 중 확인 가능한 주요 항목 예:\n  - 제안서는 반드시 직접 제출해야 하며 제출비용은 제안사 부담(문서 1).\n  - 본 사업은 제안서 보상대상에 해당하지 않아 제안서 보상을 실시하지 않음(문서 1).\n  - 소프트웨어 하자담보책임기간은 사업 종료일로부터 1년(문서 2).\n  - 소스코드·내부망 정보 등 누출금지 대상정보 및 위반 시 입찰참가자격 제한(문서 5).\n  - 사업 완료 시 감독관 입회하에 종합시험을 실시하고 규격·제안내용이 문제없이 수행되어야 함(문서 1, 4).\n- 위 문서들 어느 곳에도 “사업 총 예산”, “추정가격”, “예산안” 등의 금액 정보가 명시되어 있지 않습니다. 따라서 요청하신 예산 정보는 제공된 문서에서 확인할 수 없습니다.\n\n[출처]\n- 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp (문서 1~5)',
 'sources': ['한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp'],
 'relevance': 0.0,
 'retries': 1,
 'elapsed': 37.04,
 'errors': []}

In [35]:
# 예시 3: 첫 번째 데이터 — 제출 기한 질문
await ask("제안서 제출 마감일은 언제인가요?",
    filters={"파일": "한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp"})

--- 검색 점수 낮음(0.0000) → grade 평가 진행 ---
--- 배치 평가 시작 ---
  문서 1: no
  문서 2: no
  문서 3: no
  문서 4: no
  문서 5: no
--- 배치 검수 완료: 0/5 관련, 점수 0.00 ---
--- 관련성 부족 → 질문 재작성 ---
--- 질문 재작성 (retry 1) ---
  원래: 제안서 제출 마감일은 언제인가요?
  변환: 제안서 제출 마감일(제출기한/마감)
--- 검색 점수 낮음(0.0000) → grade 평가 진행 ---
--- 배치 평가 시작 ---
  문서 1: no
  문서 2: no
  문서 3: no
  문서 4: no
  문서 5: no
--- 배치 검수 완료: 0/5 관련, 점수 0.00 ---
--- 최대 재시도(1회) 도달 → 답변 생성 ---
핵심 요약
- 제안서 제출 마감일은 "제안서 제출 일자 : 2024.10.15.(화) 10:00~16:00"이며, 제출은 방문접수(직접제출)만 허용됩니다.

세부 사항
- 제출 기한(원문 그대로): 제안서 제출 일자 : 2024.10.15.(화) 10:00~16:00. (문서 3)
- 제출 방법: 방문접수(우편접수 및 e-mail, 팩스 등은 접수 불가). (문서 3, 문서 2)
- 제출 장소: 본관2층 사무처. (문서 3, 문서 2)
- 제출 시 요구 서류·수량 요약: 제안서 10부(원본1부 + 사본9부), 제안내용 수록 USB 1매(문서 1·3) — 제출물 준비를 마감 시간 내에 직접 제출해야 함.
- 유의: 전화문의는 법적 효력이 없고, 법적 효력 발생을 위해서는 제안서 제출 마감일로부터 5일전까지 서면 질의해야 한다는 점을 참고하세요. (문서 2)

[출처]
- 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp (문서 2, 문서 3, 문서 1, 문서 4)
Q: 제안서 제출 마감일은 언제인가요?
   필터: {'파일': '한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.

{'answer': '핵심 요약\n- 제안서 제출 마감일은 "제안서 제출 일자 : 2024.10.15.(화) 10:00~16:00"이며, 제출은 방문접수(직접제출)만 허용됩니다.\n\n세부 사항\n- 제출 기한(원문 그대로): 제안서 제출 일자 : 2024.10.15.(화) 10:00~16:00. (문서 3)\n- 제출 방법: 방문접수(우편접수 및 e-mail, 팩스 등은 접수 불가). (문서 3, 문서 2)\n- 제출 장소: 본관2층 사무처. (문서 3, 문서 2)\n- 제출 시 요구 서류·수량 요약: 제안서 10부(원본1부 + 사본9부), 제안내용 수록 USB 1매(문서 1·3) — 제출물 준비를 마감 시간 내에 직접 제출해야 함.\n- 유의: 전화문의는 법적 효력이 없고, 법적 효력 발생을 위해서는 제안서 제출 마감일로부터 5일전까지 서면 질의해야 한다는 점을 참고하세요. (문서 2)\n\n[출처]\n- 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp (문서 2, 문서 3, 문서 1, 문서 4)',
 'sources': ['한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp'],
 'relevance': 0.0,
 'retries': 1,
 'elapsed': 38.99,
 'errors': []}

In [36]:
# 예시 4: 첫 번째 데이터 — 참가 자격 질문
await ask("입찰 참가 자격 요건은 무엇인가요?",
    filters={"파일": "한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp"})

--- 검색 점수 낮음(0.0000) → grade 평가 진행 ---
--- 배치 평가 시작 ---
  문서 1: yes
  문서 2: no
  문서 3: yes
  문서 4: no
  문서 5: yes
--- 배치 검수 완료: 3/5 관련, 점수 0.60 ---
--- 관련성 충분 → 답변 생성 ---
핵심 요약
- 입찰 참가자는 “다음 요건을 모두 갖춘 사업자”여야 하며, RFP에 명시된 ①~⑦ 항목을 모두 충족해야 입찰자격이 있습니다.

세부 사항 (입찰 참가 자격 요건)
- 기본 문구
  - 본 제안 사업의 수행이 가능한 업체로서 다음 요건을 모두 갖춘 사업자

- 요구사항 항목 (원문 표기 기준으로 정리)
  1. 「국가를 당사자로 하는 계약에 관한법률 시행령」 제12조 및 동법 시행규칙 제14조 규정에 의한 경쟁입찰 참가자격을 갖춘 업체
  2. 「소프트웨어산업진흥법」 제24조 및 동법 시행령 제17조에 의한 소프트웨어사업자 (컴퓨터관련서비스업) 신고를 필한 업체
  3. 중소벤처기업부 고시 「중소기업자간 경쟁제품 직접생산확인기준」에 따라 직접생산증명서
     - [입찰마감일 전일까지 소프트웨어엔지니어링업 정보시스템개발서비스(세부품명번호: 8111159901)로 발행된 것으로 유효기간 내에 있어야 함]
  4. 최근 3년 이내에 ASP.NET으로 대학관련 종합정보/학생이력/역량시스템 구축 개발 수주가 단일건으로 1억원 이상 실적이 있는 업체
     - ※ Windows Server, MS-SQL기반 ASP.NET 개발환경에 구축 실적이 있는 업체
  5. .NET기반의 Windows Server의 Application 생성 및 표준화된 표준프레임워크을보유하고 있는 업체(타대학교에 적용 및 검증된 프레임워크 활용)
  6. 공동수급(공동이행방식) 및 하도급은 불가함의 규정에 의한 경쟁 입찰 참가 자격을 갖춘 과업 수행 가능자

- 기타 관련 유의사항(입찰자격 관련 문구)
  - 제안 핵심인력은 자사 핵심인력으로 구성하여야 함 (채용 예정 핵심인력인

{'answer': '핵심 요약\n- 입찰 참가자는 “다음 요건을 모두 갖춘 사업자”여야 하며, RFP에 명시된 ①~⑦ 항목을 모두 충족해야 입찰자격이 있습니다.\n\n세부 사항 (입찰 참가 자격 요건)\n- 기본 문구\n  - 본 제안 사업의 수행이 가능한 업체로서 다음 요건을 모두 갖춘 사업자\n\n- 요구사항 항목 (원문 표기 기준으로 정리)\n  1. 「국가를 당사자로 하는 계약에 관한법률 시행령」 제12조 및 동법 시행규칙 제14조 규정에 의한 경쟁입찰 참가자격을 갖춘 업체\n  2. 「소프트웨어산업진흥법」 제24조 및 동법 시행령 제17조에 의한 소프트웨어사업자 (컴퓨터관련서비스업) 신고를 필한 업체\n  3. 중소벤처기업부 고시 「중소기업자간 경쟁제품 직접생산확인기준」에 따라 직접생산증명서\n     - [입찰마감일 전일까지 소프트웨어엔지니어링업 정보시스템개발서비스(세부품명번호: 8111159901)로 발행된 것으로 유효기간 내에 있어야 함]\n  4. 최근 3년 이내에 ASP.NET으로 대학관련 종합정보/학생이력/역량시스템 구축 개발 수주가 단일건으로 1억원 이상 실적이 있는 업체\n     - ※ Windows Server, MS-SQL기반 ASP.NET 개발환경에 구축 실적이 있는 업체\n  5. .NET기반의 Windows Server의 Application 생성 및 표준화된 표준프레임워크을보유하고 있는 업체(타대학교에 적용 및 검증된 프레임워크 활용)\n  6. 공동수급(공동이행방식) 및 하도급은 불가함의 규정에 의한 경쟁 입찰 참가 자격을 갖춘 과업 수행 가능자\n\n- 기타 관련 유의사항(입찰자격 관련 문구)\n  - 제안 핵심인력은 자사 핵심인력으로 구성하여야 함 (채용 예정 핵심인력인 경우에는 별도로 명기하고, 계약체결 전까지 채용 완료)\n  - 적법한 파견근로자는 자사인력으로 간주하나, 원 소속사를 반드시 명기하여야 함\n\n[출처]\n- 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp (문서 1, 

In [37]:
# 예시 5: 첫 번째 데이터 — 기술 요구사항 질문
await ask("주요 기술 요구사항은 무엇인가요?",
    filters={"파일": "한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp"})

--- 검색 점수 낮음(0.0000) → grade 평가 진행 ---
--- 배치 평가 시작 ---
  문서 1: no
  문서 2: yes
  문서 3: yes
  문서 4: no
  문서 5: no
--- 배치 검수 완료: 2/5 관련, 점수 0.40 ---
--- 관련성 부족 → 질문 재작성 ---
--- 질문 재작성 (retry 1) ---
  원래: 주요 기술 요구사항은 무엇인가요?
  변환: 주요 기술요구사항? 필수사양, 기술규격
--- 검색 점수 낮음(0.0000) → grade 평가 진행 ---
--- 배치 평가 시작 ---
  문서 1: yes
  문서 2: no
  문서 3: yes
  문서 4: yes
  문서 5: yes
--- 배치 검수 완료: 4/5 관련, 점수 0.80 ---
--- 최대 재시도(1회) 도달 → 답변 생성 ---
요약
- 제공된 RFP 문서는 한영대학교 ‘트랙운영 학사정보시스템 고도화’ 사업에 대해 제안서에 반드시 포함해야 할 기술요구사항의 항목(구성도, 장비 제안, 보안, 데이터전환, 성능·테스트 등)과 평가기준(배점)을 제시하고 있으나, 개별 장비의 세부 필수사양(예: CPU, 메모리, 저장용량, 네트워크 속도 등)이나 구체적 기술규격 수치는 문서에 기재되어 있지 않습니다. (해당 세부 수치는 "제공된 문서에서 해당 정보를 확인할 수 없습니다.")

세부 사항
- 요구·작성 범위(문서 합본 기준)
  - 목표시스템 구성도 및 구성 체계 제시: 발주기관 요구사항을 만족하는 목표시스템 구성도 및 전반 구성 방안을 제시할 것. (문서1)
  - 제안 장비 내역: 규격, 기능, 수량 등 제안 장비 내역을 제시하고, 설치·납품·확장성 방안 명시할 것. (문서1, 문서3)
  - 기능 요구사항: 요청사항 구분별 상세기능 요구사항의 구현 방안을 구체적으로 기술할 것(요구사항을 만족하는지 설명). (문서1, 문서3)
  - 보안 요구사항: 설계단계부터 반영할 표준 및 구현 방안 등을 구체적으로 기술할 것. (문서1, 문서3)
 

{'answer': '요약\n- 제공된 RFP 문서는 한영대학교 ‘트랙운영 학사정보시스템 고도화’ 사업에 대해 제안서에 반드시 포함해야 할 기술요구사항의 항목(구성도, 장비 제안, 보안, 데이터전환, 성능·테스트 등)과 평가기준(배점)을 제시하고 있으나, 개별 장비의 세부 필수사양(예: CPU, 메모리, 저장용량, 네트워크 속도 등)이나 구체적 기술규격 수치는 문서에 기재되어 있지 않습니다. (해당 세부 수치는 "제공된 문서에서 해당 정보를 확인할 수 없습니다.")\n\n세부 사항\n- 요구·작성 범위(문서 합본 기준)\n  - 목표시스템 구성도 및 구성 체계 제시: 발주기관 요구사항을 만족하는 목표시스템 구성도 및 전반 구성 방안을 제시할 것. (문서1)\n  - 제안 장비 내역: 규격, 기능, 수량 등 제안 장비 내역을 제시하고, 설치·납품·확장성 방안 명시할 것. (문서1, 문서3)\n  - 기능 요구사항: 요청사항 구분별 상세기능 요구사항의 구현 방안을 구체적으로 기술할 것(요구사항을 만족하는지 설명). (문서1, 문서3)\n  - 보안 요구사항: 설계단계부터 반영할 표준 및 구현 방안 등을 구체적으로 기술할 것. (문서1, 문서3)\n  - 데이터 요구사항: 데이터 전환 계획, 검증 방법, 에러 데이터 처리 방법 등 구체적 기술 필요. (문서1, 문서3)\n  - 제약사항: 한영대학 개발환경 관련 제약을 명시하고, 이를 충족하는 구현·테스트 방안 기술. (문서1, 문서3)\n\n- 성능·품질·테스트 관련 요구\n  - 성능 요구사항: 분석·도구·테스트 방안 등을 통해 요구 성능 충족 가능성을 구체적 기술. (문서1, 문서4)\n  - 인터페이스 요구사항: 타 시스템 연계 방안(장·단점 포함) 및 사용자 인터페이스의 분석·설계·구현 방안 기술. (문서4)\n  - 테스트 요구사항: 단위/통합/시스템/성능 테스트 등 유형, 테스트 환경·방법·절차 명시. (문서1, 문서4)\n  - 품질 요구사항: 각 단계별 품질 점검·검토 방안 및 별도의 전문인력 투입 

### 8-3. 발주기관 필터 질의

특정 발주기관 소속 문서만 대상으로 질의한다.

In [38]:
# 예시 6: 발주기관 필터 — 한영대학 사업 요약
await ask("사업 범위와 주요 요구사항을 정리해줘", filters={"기관": "한영대학"})

--- 검색 점수 낮음(0.0000) → grade 평가 진행 ---
--- 배치 평가 시작 ---
  문서 1: yes
  문서 2: yes
  문서 3: yes
  문서 4: yes
  문서 5: yes
--- 배치 검수 완료: 5/5 관련, 점수 1.00 ---
--- 관련성 충분 → 답변 생성 ---
핵심 요약
- 사업명: 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화. 주요 범위는 트랙(전공/역량) 운영을 지원하는 학사정보시스템의 고도화(기능·성능·보안·운영·유지보수 포함)이며, 기능요구·보안·데이터·성능·품질·프로젝트관리·사업지원(교육·유지보수 등) 항목을 포함한 종합적 시스템 구축·납품·안정화·지식이전이 요구됩니다.
- 제출기한·방법 등 행정요건: 제안서 방문접수(우편/이메일/팩스 불가), 제안서 제출일은 "2024.10.15.(화) 10:00~16:00".
- 계약·지적재산·하자담보: S/W는 타 기관과 공동활용 계획 없음, 하자담보책임기간은 "사업을 종료한 날로부터 1년", 계약방식은 협상에 의한 계약(관련 규정 적용).
- 주요 평가·자격요건: 기능점수 기반으로 핵심인력만 투입계획 제출, 최근 3년간 대학 관련 동등 이상의 구축실적 제출 등.

세부 사항 (문서의 항목별 정리)
- 사업 범위(문서 전반)
  - 트랙운영을 포함한 학사정보시스템 고도화 전반(목표시스템 구성·구축·납품·설치·확장성 포함).
  - 개발·설계·테스트·시범운영·도입·운영안정화·지식이전·유지보수까지 포괄.

- 기능 요구사항 (문서2)
  - 발주기관 요구사항을 만족하는 목표시스템 구성도 및 구성 체계 제시.
  - 시스템 구축 전반의 구성 방안 및 특징 제시.
  - 제안 장비 내역(규격, 기능, 수량) 및 시스템 납품·설치 방안 제시.
  - 제안 장비의 확장성 제시.

- 보안·데이터·개인정보 (문서3·4)
  - 개인정보보호를 고려한 시스템 설계.
  - 공급받은 SW 산출물에 대해 누출금지정보를 삭제·활용하고, 확인을 위한 공급자 대

{'answer': '핵심 요약\n- 사업명: 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화. 주요 범위는 트랙(전공/역량) 운영을 지원하는 학사정보시스템의 고도화(기능·성능·보안·운영·유지보수 포함)이며, 기능요구·보안·데이터·성능·품질·프로젝트관리·사업지원(교육·유지보수 등) 항목을 포함한 종합적 시스템 구축·납품·안정화·지식이전이 요구됩니다.\n- 제출기한·방법 등 행정요건: 제안서 방문접수(우편/이메일/팩스 불가), 제안서 제출일은 "2024.10.15.(화) 10:00~16:00".\n- 계약·지적재산·하자담보: S/W는 타 기관과 공동활용 계획 없음, 하자담보책임기간은 "사업을 종료한 날로부터 1년", 계약방식은 협상에 의한 계약(관련 규정 적용).\n- 주요 평가·자격요건: 기능점수 기반으로 핵심인력만 투입계획 제출, 최근 3년간 대학 관련 동등 이상의 구축실적 제출 등.\n\n세부 사항 (문서의 항목별 정리)\n- 사업 범위(문서 전반)\n  - 트랙운영을 포함한 학사정보시스템 고도화 전반(목표시스템 구성·구축·납품·설치·확장성 포함).\n  - 개발·설계·테스트·시범운영·도입·운영안정화·지식이전·유지보수까지 포괄.\n\n- 기능 요구사항 (문서2)\n  - 발주기관 요구사항을 만족하는 목표시스템 구성도 및 구성 체계 제시.\n  - 시스템 구축 전반의 구성 방안 및 특징 제시.\n  - 제안 장비 내역(규격, 기능, 수량) 및 시스템 납품·설치 방안 제시.\n  - 제안 장비의 확장성 제시.\n\n- 보안·데이터·개인정보 (문서3·4)\n  - 개인정보보호를 고려한 시스템 설계.\n  - 공급받은 SW 산출물에 대해 누출금지정보를 삭제·활용하고, 확인을 위한 공급자 대표명의 확약서 제출 요구.\n  - 반출된 SW산출물을 제3자에게 제공하려면 발주기관 사전승인 필요.\n  - 누출·무단활용 시 국가계약법 및 지방계약법에 따라 입찰참가자격 제한(문서3: "국가계약법 제76조제1항제3호 및 지방계약법 제92조제1항제19호

### 8-4. 필터 없이 전체 문서 대상 질의

100개 문서 전체를 대상으로 질의하여 여러 사업 정보를 통합 답변받는다.

In [39]:
# 예시 7: 전체 문서 대상 — 협상에 의한 계약 조건
await ask("협상에 의한 계약 방식을 사용하는 사업은 어떤 것이 있나요?")

--- 검색 점수 낮음(0.0115) → grade 평가 진행 ---
--- 배치 평가 시작 ---
  문서 1: yes
  문서 2: no
  문서 3: yes
  문서 4: yes
  문서 5: yes
--- 배치 검수 완료: 4/5 관련, 점수 0.80 ---
--- 관련성 충분 → 답변 생성 ---
핵심 요약
- 제공된 문서 중 협상에 의한 계약 방식을 명시한 사업은 다음 4건입니다: 한국연구재단(기초학문자료센터), 한국사회보장정보원(라오스 보건의료정보화 사전타당성), 수협중앙회(강릉어선안전조업국 상황관제시스템), 고양도시관리공사(관산근린공원 다목적구장 홈페이지).  
- 한국발명진흥회(건설기술 특허·실용신안 관리시스템 개편) 문서에서는 “협상에 의한 계약” 명시를 확인할 수 없습니다.

세부 사항
- 한국연구재단 — 2024년 기초학문자료센터 시스템 운영 및 연구성과물 DB구축 사업
  - 관련 문구: "우선협상대상자로 선정하여 ‘협상에 의한 계약체결기준’에 따라 협상함"
  - 협상방법/기준 등 협상 관련 조항들이 명시되어 있음.

- 한국사회보장정보원 — 라오스 보건의료정보화 협력을 위한 사전타당성 조사
  - 관련 문구(협상·낙찰자 선정 항목): "협상범위는 협상대상자가 제안한 사업내용, 이행방법, 이행일정 등 제안서로 제출된 모든 내용이며 협상을 통해 그 내용을 조정할 수 있음"
  - 우선협상대상자와의 협상 성립·결렬 절차 등 협상 절차가 상세히 기재됨.

- 수협중앙회 — 강릉어선안전조업국 상황관제시스템 구축
  - 관련 문구: "계약방법 : 협상에 의한 계약"
  - 또한 "기술평가와 가격평가 후 협상에 의한 계약방식 추진" 등 협상방식 및 평가·선정절차 명시.

- 고양도시관리공사 — 관산근린공원 다목적구장 홈페이지 및 회원 통합운영 관리 시스템 구축 [협상에 의한 계약]
  - 문서 제목 및 본문에 협상 관련 규정(협상진행, 가격협상, 협상기간 등)이 포함되어 있음.

- 한국발명진흥회 — 2024년 건설기술에 관한 특허·실용신안 활용

{'answer': '핵심 요약\n- 제공된 문서 중 협상에 의한 계약 방식을 명시한 사업은 다음 4건입니다: 한국연구재단(기초학문자료센터), 한국사회보장정보원(라오스 보건의료정보화 사전타당성), 수협중앙회(강릉어선안전조업국 상황관제시스템), 고양도시관리공사(관산근린공원 다목적구장 홈페이지).  \n- 한국발명진흥회(건설기술 특허·실용신안 관리시스템 개편) 문서에서는 “협상에 의한 계약” 명시를 확인할 수 없습니다.\n\n세부 사항\n- 한국연구재단 — 2024년 기초학문자료센터 시스템 운영 및 연구성과물 DB구축 사업\n  - 관련 문구: "우선협상대상자로 선정하여 ‘협상에 의한 계약체결기준’에 따라 협상함"\n  - 협상방법/기준 등 협상 관련 조항들이 명시되어 있음.\n\n- 한국사회보장정보원 — 라오스 보건의료정보화 협력을 위한 사전타당성 조사\n  - 관련 문구(협상·낙찰자 선정 항목): "협상범위는 협상대상자가 제안한 사업내용, 이행방법, 이행일정 등 제안서로 제출된 모든 내용이며 협상을 통해 그 내용을 조정할 수 있음"\n  - 우선협상대상자와의 협상 성립·결렬 절차 등 협상 절차가 상세히 기재됨.\n\n- 수협중앙회 — 강릉어선안전조업국 상황관제시스템 구축\n  - 관련 문구: "계약방법 : 협상에 의한 계약"\n  - 또한 "기술평가와 가격평가 후 협상에 의한 계약방식 추진" 등 협상방식 및 평가·선정절차 명시.\n\n- 고양도시관리공사 — 관산근린공원 다목적구장 홈페이지 및 회원 통합운영 관리 시스템 구축 [협상에 의한 계약]\n  - 문서 제목 및 본문에 협상 관련 규정(협상진행, 가격협상, 협상기간 등)이 포함되어 있음.\n\n- 한국발명진흥회 — 2024년 건설기술에 관한 특허·실용신안 활용실적 관리시스템 개편 용역\n  - 제공된 문서에서는 입찰·제안서 효력, 보안확약서, 용역대가 지급 등 내용은 있으나, 문서에서 "협상에 의한 계약" 방식 사용 여부를 명시적으로 확인할 수 없습니다.\n  - 따라서 해당 사업이 협상에 의한 계약 방식

### 8-5. 대화형 Q&A

`input()`을 통해 자유롭게 질문할 수 있다. `q` 또는 `종료`를 입력하면 종료된다.

In [40]:
# 대화형 Q&A 루프 (비동기)
print("=" * 60)
print("  입찰메이트 Q&A — 100개 RFP 문서 질의응답")
print("  종료: 'q' 또는 '종료' 입력")
print("  필터 사용: '필터:기관=한영대학' 형태로 질문 앞에 추가")
print("=" * 60)

while True:
    user_input = input("\n질문> ").strip()
    if not user_input or user_input.lower() in ("q", "quit", "exit", "종료"):
        print("Q&A를 종료합니다.")
        break

    # 필터 파싱: "필터:기관=한영대학 질문내용" 형태 지원
    filters = {}
    question = user_input
    if user_input.startswith("필터:"):
        parts = user_input.split(" ", 1)
        filter_str = parts[0].replace("필터:", "")
        question = parts[1] if len(parts) > 1 else ""
        for pair in filter_str.split(","):
            if "=" in pair:
                key, val = pair.split("=", 1)
                filters[key.strip()] = val.strip()

    if not question:
        print("질문을 입력해주세요.")
        continue

    print("-" * 60)
    await ask(question, filters=filters if filters else None)
    print("-" * 60)

  입찰메이트 Q&A — 100개 RFP 문서 질의응답
  종료: 'q' 또는 '종료' 입력
  필터 사용: '필터:기관=한영대학' 형태로 질문 앞에 추가
Q&A를 종료합니다.
